# FedBaselineNet — FedAvg vs. FedProx
## Federated Multi-Hospital Brain Tumor MRI Classification (SMGF-Net + MS-CA-ECA backbone)

**Scope of this notebook:** a clean two-way comparison of the classic
federated aggregation baselines — **FedAvg** and **FedProx** — on
**non-IID (label-distribution-skewed) client splits only**. Client data is
partitioned with a Dirichlet(α) allocation over class labels (Hsu et al.,
2019) rather than the stratified/balanced split — each simulated hospital
ends up with a different, skewed mix of tumor classes, the harder and more
clinically realistic regime this literature review's FedAvg weaknesses
section warns about. No custom aggregation method, no significance
testing, no ablations. The classifier itself (SpatialBackbone + MorphologyBranch
→ GatedFusion2 → MS-CA-ECA attention → SmallHead, <=211,560-parameter budget,
CAWL+ loss) is reused verbatim from the SMGF-Net + MS-CA-ECA notebook — the
old NeuroRaceNet V2 backbone (AMFF → RLDA → ProjectionHead) has been removed
entirely.

**Added in this version:** full post-hoc test-set diagnostics for each
strategy's final global model — classification report, confusion matrix,
ROC curves, PR curves, t-SNE of the embedding space, and explainability
(Grad-CAM++, Score-CAM, LIME).

## Rerun guide (what this revision of the notebook changes)

Every default below reproduces the setup that produced the numbers currently in the paper, **except** the DP
noise calibration (§24.6), which is a bug fix and therefore changes the achieved epsilon.

| Reviewer concern | What to run / flip | Where |
|---|---|---|
| Non-private GroupNorm control (DP cost vs. normalization cost) | `RUN_GN_CONTROL = True` (default now) | §24.2, run in §24.9 |
| Clipping vs. noise (optional) | `RUN_CLIP_ONLY_CONTROL = True` | §24.2, run in §24.9 |
| Solver inputs differ from what the Opacus loader really does (achieved epsilon above target) | fixed in `run_dp_federated_experiment`; the run asserts the match and reports `epsilon / target` | §24.6 |
| Federated and centralized recipes use different AdamW settings | `MATCH_CENTRALIZED_HPARAMS = True` (lr 1.8e-4, wd 3e-4) | §2 |
| Accuracy still changing at round 50 | raise `COMM_ROUNDS` (baseline only; `DP_ROUNDS` is separate, default 50) | §2, §24.2 |
| FedAvg vs FedProx from a single run | `FL_SEEDS = [42, 13, 77]` (seed changes init, Dirichlet draw and client hold-out) | §2, §15 |
| Single FedProx mu | `RUN_MU_SWEEP = True`, `FEDPROX_MU_GRID` | §2, §15.1 |
| Per-class tables, tumor-vs-no-tumor errors, saved predictions | written automatically for every run: `*_per_class.csv`, `*_confusion.csv`, `*_test_predictions.npz`, `*_global_model.pt`, `*_history.csv` | §14.1, §15, §25.2 |
| Heterogeneity (JSD / EMD) and client sizes after the 15 % local hold-out | `partition_heterogeneity()` on the Dirichlet partition (§24.2a) | §14.1, §24.7 |

**About the global test set.** There is no server-side validation set in this pipeline. Each client holds out 15 % of
its own images and uses them for monitoring only. The global test set is scored every round so the curves can be
drawn, but nothing is selected with it: the number reported is always the last round.

**Outputs of the previous version of this notebook were cleared.** Run top to bottom, or use the §23.9 replay flow
for DP-only sessions as before. Runs saved by an older version are *not* reused by the epsilon ablation, because they
lack the `solver_matches_loader` flag.

**Non-IID specifics.** `FL_SEEDS[0]` must equal `SEED` (42): that seed reproduces the partition in the paper
(client sizes 307 / 543 / 2152 / 1885 / 68). Any further seed draws a *new* Dirichlet(0.3) partition, so the
across-seed spread includes partition variability, not only initialisation. The DP arm always runs on the first
seed's partition. The solver now reads q and steps from the Opacus loader; with it the 68-image client (57 after the
hold-out, q = 0.5) is calibrated for 2 steps per epoch, not 1.


## §1 · Dependencies

In [ ]:
# Kaggle already ships numpy/scipy/scikit-learn/pandas that are ABI-matched to each other.
# Upgrading any one of them (via -U) can pull in a numpy/scipy pair that no longer matches
# the pre-built scikit-learn/scipy binaries, corrupting numpy ("No module named 'numpy.char'").
# Fix: only install what's actually missing, never touch numpy/scipy/scikit-learn/pandas,
# and drop flwr (this notebook implements FedAvg/FedProx manually -- flwr is never
# imported anywhere below, so installing it is unnecessary risk for no benefit).
# timm is no longer needed either -- the SMGF-Net + MS-CA-ECA backbone is a from-scratch
# depthwise-separable network, not a timm/pretrained model.
#
# If you already ran the OLD version of this cell and hit "No module named 'numpy.char'",
# reinstalling correctly will NOT fix an already-corrupted session -- restart the kernel first
# (Kaggle: Run -> Restart session), then re-run the notebook from the top.
!pip install -q scikit-image lime


## §2 · Config

In [ ]:
import os, math, random, warnings, hashlib, copy, itertools
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                              precision_score, recall_score, roc_auc_score,
                              confusion_matrix, classification_report,
                              matthews_corrcoef, cohen_kappa_score,
                              roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE    = 224
NUM_CLASSES = 4
CLASS_NAMES  = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".jfif", ".tif", ".tiff", ".webp"}

OUT_DIR  = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 32
NUM_WORKERS = 4          # OpenCV/CLAHE preprocessing is CPU-bound, too few workers starves the GPU

DROPOUT_RATE  = 0.3
USE_AMP       = True
# AMEF (adaptive gamma + entropy-gated CLAHE). The centralized stage-1 notebook (96.28 %) does NOT apply AMEF, so the default here is False
# to give one preprocessing pipeline across the whole paper. Set True to reproduce the earlier AMEF-on federated/DP runs.
USE_AMEF      = False

# --- Federated-learning-specific config -----------------------------------
NUM_CLIENTS          = 5          # 5 simulated "hospitals"
COMM_ROUNDS          = 50         # baseline rounds; raise (e.g. 100) to check convergence
LOCAL_EPOCHS         = 2          # local SGD epochs per round (communication-efficient)
# --- Rerun controls. Every default reproduces the recorded setup. -------------
MATCH_CENTRALIZED_HPARAMS = False   # True -> client AdamW lr 1.8e-4 / wd 3e-4 (the centralized recipe)
FL_SEEDS                  = [SEED]  # e.g. [42, 13, 77]; the first entry must equal SEED (DP reuses its partition)
RUN_MU_SWEEP              = False   # FedProx mu grid on FL_SEEDS[0]
FEDPROX_MU_GRID           = [0.001, 0.01, 0.1]

CLIENT_LR            = 1.8e-4 if MATCH_CENTRALIZED_HPARAMS else 1.2e-4
CLIENT_WEIGHT_DECAY  = 3e-4   if MATCH_CENTRALIZED_HPARAMS else 2e-4
FEDPROX_MU           = 0.01       # proximal term strength for FedProx
NONIID_ALPHA          = 0.3       # Dirichlet concentration for label-skew client split
                                   # (Hsu et al., 2019) -- smaller = more skewed/non-IID,
                                   # larger -> IID as alpha -> infinity. 0.3 is a commonly
                                   # used "moderately-to-strongly non-IID" setting.
BN_LAYER_KEYWORDS    = ("bn", "running_mean", "running_var", "num_batches_tracked")

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True   # fixed IMG_SIZE input -> lets cuDNN autotune conv algorithms

print("Device:", DEVICE)
print("Classes:", CLASS_NAMES)
print(f"NUM_CLIENTS={NUM_CLIENTS}  COMM_ROUNDS={COMM_ROUNDS}  LOCAL_EPOCHS={LOCAL_EPOCHS}  "
      f"CLIENT_LR={CLIENT_LR:g}  WD={CLIENT_WEIGHT_DECAY:g}  FL_SEEDS={FL_SEEDS}")
def _loader_kwargs(shuffle: bool) -> dict:
    """Shared DataLoader kwargs: pin_memory speeds up host->GPU transfer,
    persistent_workers avoids re-spawning worker processes every epoch/round
    (both matter here because each federated round creates fresh DataLoaders)."""
    return dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                pin_memory=(DEVICE == "cuda"),
                persistent_workers=(NUM_WORKERS > 0))


## §3 · Literature Review — FedAvg and FedProx

### §3.1 FedAvg (McMahan et al., 2017)
**Mechanism:** each round, a fraction of clients run several local SGD epochs on
their private data; the server averages client weights, weighted by local
dataset size (`n_k / n_total`).

- **Strengths:** simple, minimal communication (one weight vector per round),
  well understood convergence under IID data, no extra hyperparameters.
- **Weaknesses:** assumes (near-)IID data. Under label-distribution skew or
  quantity skew, local updates diverge ("client drift") and simple averaging
  can cancel out useful gradient directions.
- **Computational complexity:** O(local epochs x batches) per client per
  round; server cost is O(1) weighted average.
- **Communication cost:** one full model download + upload per participating
  client per round -- for our <=211,560-parameter model this is ~0.81 MB (fp32)
  each way, i.e. ~1.6 MB/client/round.
- **Convergence:** provably converges for convex objectives under IID data;
  under non-IID data convergence bounds degrade with the degree of
  heterogeneity (Li et al., 2020, "On the Convergence of FedAvg on Non-IID Data").

### §3.2 FedProx (Li et al., 2020)
**Mechanism:** adds a proximal term `(mu/2)||w - w_global||^2` to each
client's local loss, constraining local updates from drifting too far from
the global model; otherwise identical communication pattern to FedAvg.

- **Strengths:** bounded client drift under non-IID data and partial
  participation / stragglers; retains convergence guarantees FedAvg loses
  under heterogeneity; single scalar hyperparameter (mu).
- **Weaknesses:** the proximal term is global and static -- it penalizes
  every client's deviation equally regardless of data quality; mu must be
  tuned per task.
- **Computational complexity:** local step adds one extra term (O(d) per
  step) -- negligible overhead relative to backbone forward/backward cost.
- **Communication cost:** identical to FedAvg (no extra payload).

### §3.3 FedBN (Li et al., 2021) — background only, not evaluated

Retained as related work. FedBN is **not** one of the strategies run in
§15: it is incompatible with the DP arm (BatchNorm is removed under
DP-SGD, and local BN statistics are the un-noised private quantity DP
protects), so it was dropped from the comparison.

**Mechanism:** BatchNorm layer statistics/parameters are kept local and
never communicated or averaged; only non-BN parameters are federated. This
directly targets feature-distribution shift (domain shift), since BN
statistics absorb most of the domain-specific shift in CNNs.

- **Strengths:** explicitly addresses feature-distribution shift -- highly
  relevant when hospitals use different scanners; no proximal-term tuning;
  empirically strong on multi-institution medical imaging benchmarks.
- **Weaknesses:** addresses feature shift but not label shift or quantity
  skew; local BN stats mean each client effectively holds a domain-specific
  model.
- **Computational complexity:** identical to FedAvg (BN stats are cheap:
  MobileNetV3-Small backbone here has ~34 BN layers, a negligible fraction of
  total FLOPs).
- **Communication cost:** slightly lower than FedAvg -- BN parameters are
  excluded from the upload/download payload.

## §4 · Preprocessing (AMEF) — reused verbatim (unchanged across model versions)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Cannot read: {path}")
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY if img.shape[-1] == 3
                           else cv2.COLOR_BGRA2GRAY)
    return img

def robust_normalize(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, [1, 99])
    img = np.clip(img, p1, p99)
    img -= img.min(); img /= (img.max() + 1e-8)
    return img

def brain_mask(img01: np.ndarray) -> np.ndarray:
    img_u8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)
    _, mask = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num <= 1:
        return np.ones_like(img01, dtype=bool)
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    brain = (labels == largest)
    if brain.sum() < 0.02 * img01.size:
        return np.ones_like(img01, dtype=bool)
    return brain

def laplacian_noise_estimate(gray_u8: np.ndarray) -> float:
    H, W = gray_u8.shape
    if H < 3 or W < 3:
        return 0.0
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = cv2.filter2D(gray_u8.astype(np.float32), -1, kernel)
    sigma = np.sqrt(np.pi / 2) * np.mean(np.abs(conv)) / (6 * (W - 2) * (H - 2) + 1e-8) * (W * H)
    return float(np.clip(sigma, 0.0, 50.0))

def adaptive_gamma(img01: np.ndarray, mask: np.ndarray, target: float = 0.5) -> float:
    vals = img01[mask]
    if vals.size == 0:
        return 1.0
    mu = float(np.clip(vals.mean(), 1e-3, 1 - 1e-3))
    gamma = np.log(target) / np.log(mu)
    return float(np.clip(gamma, 0.5, 2.0))

def masked_entropy(img01: np.ndarray, mask: np.ndarray, bins: int = 32) -> float:
    vals = img01[mask]
    if vals.size == 0:
        return 0.0
    hist, _ = np.histogram(vals, bins=bins, range=(0, 1), density=True)
    hist = hist / (hist.sum() + 1e-8)
    hist = hist[hist > 0]
    return float(-np.sum(hist * np.log2(hist)))

def amef_preprocess(img01: np.ndarray, base_clip: float = 2.0, kappa: float = 0.05,
                     entropy_skip_threshold: float = 3.6) -> np.ndarray:
    mask = brain_mask(img01)
    gray_u8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)
    sigma_noise = laplacian_noise_estimate(gray_u8)
    gamma = adaptive_gamma(img01, mask)
    img_gamma = np.clip(img01 ** gamma, 0, 1)
    ent = masked_entropy(img_gamma, mask)
    if ent < entropy_skip_threshold:
        clip_limit = base_clip / (1.0 + kappa * sigma_noise)
        clahe = cv2.createCLAHE(clipLimit=max(clip_limit, 0.5), tileGridSize=(8, 8))
        img_u8 = (img_gamma * 255).astype(np.uint8)
        img_out = clahe.apply(img_u8).astype(np.float32) / 255.0
    else:
        img_out = img_gamma
    return img_out

def to_3ch_tensor(img01: np.ndarray) -> torch.Tensor:
    rgb = np.stack([img01, img01, img01], axis=2)
    rgb = torch.tensor(rgb).permute(2, 0, 1).float()
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (rgb - mean) / std

def preprocess(path: str) -> torch.Tensor:
    gray = read_gray(path)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img01 = robust_normalize(gray)
    if USE_AMEF:
        img01 = amef_preprocess(img01)
    return to_3ch_tensor(img01)

print("Preprocessing + AMEF ready (unchanged).")


## §5 · Augmentation (training split only) — reused verbatim

In [ ]:
import torchvision.transforms.functional as TF

class MRIAugment:
    def __init__(self, p_flip=0.5, max_rotate=10.0, p_rotate=0.5,
                 brightness=0.2, contrast=0.2, p_jitter=0.5,
                 translate=0.06, p_translate=0.3,
                 use_gamma=True, gamma_range=(0.85, 1.15), p_gamma=0.3):
        self.p_flip = p_flip
        self.max_rotate = max_rotate; self.p_rotate = p_rotate
        self.brightness = brightness; self.contrast = contrast; self.p_jitter = p_jitter
        self.translate = translate; self.p_translate = p_translate
        self.use_gamma = use_gamma; self.gamma_range = gamma_range; self.p_gamma = p_gamma

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        if random.random() < self.p_flip:
            img = TF.hflip(img)
        if random.random() < self.p_rotate:
            angle = random.uniform(-self.max_rotate, self.max_rotate)
            img = TF.rotate(img, angle)
        if random.random() < self.p_translate:
            tx = random.uniform(-self.translate, self.translate) * IMG_SIZE
            ty = random.uniform(-self.translate, self.translate) * IMG_SIZE
            img = TF.affine(img, angle=0, translate=[int(tx), int(ty)], scale=1.0, shear=0)
        if random.random() < self.p_jitter:
            b = 1.0 + random.uniform(-self.brightness, self.brightness)
            c = 1.0 + random.uniform(-self.contrast, self.contrast)
            img = torch.clamp(img * c + (b - 1.0), -3.0, 3.0)
        if self.use_gamma and random.random() < self.p_gamma:
            gamma = random.uniform(*self.gamma_range)
            img = torch.sign(img) * (torch.abs(img) ** gamma)
        return img

train_augment = MRIAugment()
print("MRIAugment ready.")


## §6 · Dataset Class — reused verbatim

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, samples: List[Dict], is_train: bool = False,
                 augmenter: Optional[MRIAugment] = None):
        self.samples   = samples
        self.is_train  = is_train
        self.augmenter = augmenter

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = preprocess(s["path"])
        if self.is_train and self.augmenter is not None:
            img = self.augmenter(img)
        return {"image": img, "label": torch.tensor(s["label"], dtype=torch.long),
                "class_name": s["class_name"], "path": s["path"]}

print("BrainTumorDataset ready.")


## §7 · SMGF-Net + MS-CA-ECA — Spatial Branch, Morphology Branch, Gated Fusion, MS-CA-ECA Attention, Full Model
**Reused verbatim, byte-for-byte, from the SMGF-Net + MS-CA-ECA notebook.** The old
NeuroRaceNet V2 backbone (timm MobileNetV3-Small → AMFF → RLDA → ProjectionHead)
has been removed entirely, along with the `RLDA` and `AMFF` classes and the
`timm` dependency. `SMGFNetMSCAECA` is the only model class used by this
federated-learning frame:

Input MRI → `SpatialBackbone` (from-scratch depthwise-separable stem + 3
stages, exposing F1/F2/F3) → `{MultiScaleSpatial(Fs), MorphologyBranch(Fm)}`
→ `GatedFusion2` → **MS-CA-ECA** attention (`MultiScaleContext` + `CoordinateAttention`
+ `ECA`, summed/applied in sequence with a residual) → `SmallHead` → 4-class
logits + 128-d embedding.

`forward()` returns a dict with `"logits"` and `"emb"` — the same interface
the rest of this notebook (loss, local training/eval, Grad-CAM++, t-SNE)
already expects, so no downstream federated-learning code needed to change
beyond the model constructor call itself.

In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., 2020). ~5-9 parameters."""
    def __init__(self, channels: int, gamma: int = 2, b: int = 1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)


class DWSepBlock(nn.Module):
    """Depthwise-separable conv block: DW3x3 -> BN -> act -> PW1x1 -> BN -> act,
    with an optional stride-2 downsample and a residual connection when shapes match."""
    def __init__(self, c_in: int, c_out: int, stride: int = 1, act=nn.SiLU):
        super().__init__()
        self.dw = nn.Conv2d(c_in, c_in, 3, stride=stride, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, c_out, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.act = act(inplace=True)
        self.use_res = (stride == 1 and c_in == c_out)

    def forward(self, x):
        identity = x
        x = self.act(self.bn1(self.dw(x)))
        x = self.bn2(self.pw(x))
        if self.use_res:
            x = x + identity
        return self.act(x)


class SpatialBackbone(nn.Module):
    """Lightweight from-scratch depthwise-separable stem + 3 stages. Exposes
    F1/F2/F3 (stage outputs, strides /4, /8, /16) for multi-scale fusion --
    deliberately NOT a pretrained off-the-shelf backbone."""
    def __init__(self, stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stem_ch),
            nn.SiLU(inplace=True),
        )
        self.stage1 = self._make_stage(stem_ch, c1, depth[0])
        self.stage2 = self._make_stage(c1, c2, depth[1])
        self.stage3 = self._make_stage(c2, c3, depth[2])
        self.out_channels = [c1, c2, c3]

    @staticmethod
    def _make_stage(c_in, c_out, n_extra):
        layers = [DWSepBlock(c_in, c_out, stride=2)]
        for _ in range(n_extra):
            layers.append(DWSepBlock(c_out, c_out, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)          # /2
        f1 = self.stage1(x)       # /4
        f2 = self.stage2(f1)      # /8
        f3 = self.stage3(f2)      # /16
        return f1, f2, f3


class MultiScaleSpatial(nn.Module):
    """Spatial representation Fs: projects F1/F2/F3 to a common width, aligns
    to F3's resolution, fuses with learned global softmax weights, then a
    depthwise residual refine."""
    def __init__(self, in_channels_list: List[int], out_channels: int):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1, bias=False) for c in in_channels_list
        ])
        self.bn = nn.ModuleList([nn.BatchNorm2d(out_channels) for _ in in_channels_list])
        self.scale_weight = nn.Parameter(torch.ones(len(in_channels_list)))
        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, feats: List[torch.Tensor]) -> torch.Tensor:
        target_hw = feats[-1].shape[-2:]
        aligned = [F.adaptive_avg_pool2d(bn(p(f)), target_hw)
                   for p, bn, f in zip(self.proj, self.bn, feats)]
        alpha = torch.softmax(self.scale_weight, dim=0)
        fused = sum(a * x for a, x in zip(alpha, aligned))
        return fused + self.refine(fused)


class MorphologyBranch(nn.Module):
    """Morphology-aware representation Fm. A fixed, parameter-free
    morphological-gradient operator (dilation - erosion via max-pool) applied
    to F3, then a learnable depthwise+pointwise transform, trained end-to-end."""
    def __init__(self, c_in: int, out_channels: int):
        super().__init__()
        self.pool = nn.MaxPool2d(3, stride=1, padding=1)
        self.dw = nn.Conv2d(c_in, c_in, 3, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        dilation = self.pool(x)
        erosion = -self.pool(-x)
        grad = dilation - erosion          # morphological gradient, parameter-free
        x = self.act(self.bn1(self.dw(grad)))
        x = self.act(self.bn2(self.pw(x)))
        return x


class GatedFusion2(nn.Module):
    """Two-branch per-channel gated fusion (Spatial + Morphology). A
    per-channel, per-sample gate learned from both branches' pooled
    statistics -- gives the network the capacity to weight branches
    differently per output channel and per input image."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        hidden = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.Linear(channels * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels * 2),
        )

    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        b, c = fs.shape[0], fs.shape[1]
        pooled = torch.cat([self.pool(fs).flatten(1), self.pool(fm).flatten(1)], dim=1)
        raw = self.gate(pooled).view(b, 2, c)
        weights = torch.softmax(raw, dim=1)          # per-channel, per-sample, sums to 1 over the 2 branches
        fused = (weights[:, 0, :].view(b, c, 1, 1) * fs +
                 weights[:, 1, :].view(b, c, 1, 1) * fm)
        return fused, weights


class SimpleAvgFusion(nn.Module):
    """Parameter-free 50/50 averaging fusion. Kept for parity with the source
    notebook (used there only in an ablation variant); not exercised by this
    federated-learning frame, which always trains the full model."""
    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        fused = 0.5 * fs + 0.5 * fm
        b, c = fs.shape[0], fs.shape[1]
        weights = torch.full((b, 2, c), 0.5, device=fs.device)
        return fused, weights


class SmallHead(nn.Module):
    """GAP -> small projection -> BatchNorm1d -> GELU -> Dropout -> Linear(4)."""
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.gap(x).flatten(1)
        emb = self.act(self.bn(self.fc1(x)))
        out = self.fc2(self.drop(emb))
        return out, emb


class MultiScaleContext(nn.Module):
    """Cheap multi-scale contextual extraction: channel-reduce with 1x1, three
    parallel DEPTHWISE 3x3 convs at dilation 1/2/3 (receptive fields 3/5/7),
    concat, expand back with 1x1. No full (non-depthwise, non-1x1) convs, per
    the lightweight-operations requirement."""
    def __init__(self, channels: int, reduction: int = 18):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn_reduce = nn.BatchNorm2d(hidden)
        self.dw1 = nn.Conv2d(hidden, hidden, 3, padding=1, dilation=1, groups=hidden, bias=False)
        self.dw2 = nn.Conv2d(hidden, hidden, 3, padding=2, dilation=2, groups=hidden, bias=False)
        self.dw3 = nn.Conv2d(hidden, hidden, 3, padding=3, dilation=3, groups=hidden, bias=False)
        self.bn_cat = nn.BatchNorm2d(hidden * 3)
        self.expand = nn.Conv2d(hidden * 3, channels, 1, bias=False)
        self.bn_expand = nn.BatchNorm2d(channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        r = self.act(self.bn_reduce(self.reduce(x)))
        m = torch.cat([self.dw1(r), self.dw2(r), self.dw3(r)], dim=1)
        m = self.act(self.bn_cat(m))
        return self.bn_expand(self.expand(m))


class CoordinateAttention(nn.Module):
    """Coordinate Attention (Hou et al., 2021), lightweight form: separately
    pools along H and W to preserve direction-specific information, encodes
    both with a shared 1x1 reduction, then splits back into an H-gate and a
    W-gate (both 1x1 projections, no full convolutions)."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn = nn.BatchNorm2d(hidden)
        self.act = nn.SiLU(inplace=True)
        self.conv_h = nn.Conv2d(hidden, channels, 1, bias=False)
        self.conv_w = nn.Conv2d(hidden, channels, 1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        x_h = self.pool_h(x)                          # (b,c,h,1) -- height-direction info
        x_w = self.pool_w(x).permute(0, 1, 3, 2)       # (b,c,w,1) -- width-direction info
        y = torch.cat([x_h, x_w], dim=2)               # (b,c,h+w,1)
        y = self.act(self.bn(self.reduce(y)))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)                  # back to (b,hidden,1,w)
        g_h = torch.sigmoid(self.conv_h(y_h))          # (b,c,h,1)
        g_w = torch.sigmoid(self.conv_w(y_w))          # (b,c,1,w)
        return x * g_h * g_w


class MSCAECA(nn.Module):
    """Multi-Scale Coordinate-ECA Attention (proposed). Fused feature ->
    parallel {MultiScaleContext, CoordinateAttention} -> ECA channel
    refinement -> residual. use_multiscale / use_coord / use_eca gate each
    stage independently (identity pass-through when off); this federated
    frame always uses the full module (all three True)."""
    def __init__(self, channels: int, ms_reduction: int = 18, coord_reduction: int = 16,
                 use_multiscale: bool = True, use_coord: bool = True, use_eca: bool = True):
        super().__init__()
        self.use_multiscale = use_multiscale
        self.use_coord = use_coord
        self.use_eca = use_eca
        self.ms = MultiScaleContext(channels, reduction=ms_reduction) if use_multiscale else None
        self.coord = CoordinateAttention(channels, reduction=coord_reduction) if use_coord else None
        self.eca = ECA(channels) if use_eca else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = x
        if self.ms is not None:
            out = out + self.ms(out)
        if self.coord is not None:
            out = self.coord(out)
        if self.eca is not None:
            out = self.eca(out)
        return out + identity


# Channel configuration -- rep_width narrowed from SMGF-Net's 160 to 148 to make
# room for MS-CA-ECA under the same 211,560-parameter ceiling as the reference
# (plain-ECA) SMGF-Net. Spatial branch / Morphology branch / Gated Fusion configs
# are otherwise IDENTICAL to the reference SMGF-Net.
SMGF_MSCAECA_CONFIG = dict(stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2),
                            rep_width=148, head_hidden=128)
MSCAECA_CONFIG = dict(ms_reduction=18, coord_reduction=16)
PARAM_BUDGET = 211_560               # hard constraint: MS-CA-ECA SMGF-Net <= this


class SMGFNetMSCAECA(nn.Module):
    """SMGF-Net + MS-CA-ECA -- the only model class used throughout this
    federated-learning notebook (the old NeuroRaceNet V2 backbone and its
    RLDA/AMFF/ProjectionHead modules have been removed entirely).

    Input MRI -> SpatialBackbone (stem + 3 depthwise-separable stages,
    exposing F1/F2/F3) -> {MultiScaleSpatial(Fs), MorphologyBranch(Fm)} ->
    GatedFusion2 -> MS-CA-ECA -> SmallHead -> 4-class logits.
    """
    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = DROPOUT_RATE,
                 stem_ch: int = 32, c1: int = 48, c2: int = 88, c3: int = 160,
                 depth=(1, 2, 2), rep_width: int = 148, head_hidden: int = 128,
                 use_spatial: bool = True, use_morphology: bool = True,
                 use_gated_fusion: bool = True, attention: str = "msca_eca",
                 ms_kwargs: Optional[Dict[str, Any]] = None):
        super().__init__()
        assert use_spatial or use_morphology, "at least one of Spatial/Morphology must be active"
        self.use_spatial = use_spatial
        self.use_morphology = use_morphology
        self.use_gated_fusion = use_gated_fusion
        self.attention_type = attention
        ms_kwargs = {**MSCAECA_CONFIG, **(ms_kwargs or {})}

        self.backbone = SpatialBackbone(stem_ch, c1, c2, c3, depth=depth)
        self.multiscale = MultiScaleSpatial(self.backbone.out_channels, rep_width) if use_spatial else None
        self.morphology = MorphologyBranch(c3, rep_width) if use_morphology else None

        if use_spatial and use_morphology:
            self.fusion = GatedFusion2(channels=rep_width) if use_gated_fusion else SimpleAvgFusion()
        else:
            self.fusion = None   # single-branch ablation: nothing to fuse

        if attention == "msca_eca":
            self.attn = MSCAECA(rep_width, **ms_kwargs)
        elif attention == "eca":
            self.attn = ECA(rep_width)
        elif attention == "none":
            self.attn = nn.Identity()
        else:
            raise ValueError(f"unknown attention type: {attention}")

        self.head = SmallHead(rep_width, head_hidden, num_classes, dropout)

    def forward(self, x: torch.Tensor, return_all: bool = False) -> Dict[str, torch.Tensor]:
        f1, f2, f3 = self.backbone(x)
        fs = self.multiscale([f1, f2, f3]) if self.use_spatial else None
        fm = self.morphology(f3) if self.use_morphology else None

        weights = None
        if self.use_spatial and self.use_morphology:
            fused, weights = self.fusion(fs, fm)
        elif self.use_spatial:
            fused = fs
        else:
            fused = fm

        attn = self.attn(fused)
        logits, emb = self.head(attn)
        out = {"logits": logits, "emb": emb}
        if return_all:
            out.update({"f1": f1, "f2": f2, "f3": f3, "spatial": fs, "morphology": fm,
                        "fusion_weights": weights, "fused": fused, "attn": attn})
        return out


def build_smgf_mscaeca(**kwargs) -> SMGFNetMSCAECA:
    """Builds SMGF-Net + MS-CA-ECA. Any of SMGFNetMSCAECA's constructor args
    can be overridden via kwargs. This is the model-builder used everywhere
    in the federated-learning frame (replaces the old build_neuroracenet_v2)."""
    cfg = {**SMGF_MSCAECA_CONFIG}
    ms_override = kwargs.pop("ms_kwargs", None)
    cfg.update(kwargs)
    if ms_override is not None:
        cfg["ms_kwargs"] = ms_override
    model = SMGFNetMSCAECA(**cfg).to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"SMGF-Net + MS-CA-ECA  total={total:,}  trainable={trainable:,}  "
          f"budget_ok(<={PARAM_BUDGET:,})={trainable <= PARAM_BUDGET}")
    return model


def param_breakdown(model: "SMGFNetMSCAECA") -> Dict[str, int]:
    def count(m):
        return 0 if m is None else sum(p.numel() for p in m.parameters() if p.requires_grad)
    parts = {
        "backbone (stem + 3 DW-sep stages)": count(model.backbone),
        "spatial branch (F1/F2/F3 -> Fs)": count(model.multiscale),
        "morphology branch (Fm)": count(model.morphology),
        "gated fusion (Fs, Fm -> fused)": count(model.fusion),
        "MS-CA-ECA attention": count(model.attn),
        "classifier head": count(model.head),
    }
    parts["TOTAL"] = sum(v for k, v in parts.items())
    return parts


print("SMGFNetMSCAECA ready -- this is the ONLY model class used throughout this notebook "
      "(NeuroRaceNet V2 / AMFF / RLDA / ProjectionHead removed).")


## §8 · CAWL+ — Calibration-Aware Loss — reused verbatim

In [ ]:
def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        count = mask.sum()
        if count == 0:
            bin_accs.append(0); bin_confs.append(0); bin_counts.append(0)
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (count / len(labels)) * abs(bin_acc - bin_conf)
        bin_accs.append(bin_acc); bin_confs.append(bin_conf); bin_counts.append(count)
    return ece, bin_accs, bin_confs, bin_counts

def maximum_calibration_error(bin_accs, bin_confs, bin_counts) -> float:
    gaps = [abs(a - c) for a, c, n in zip(bin_accs, bin_confs, bin_counts) if n > 0]
    return float(max(gaps)) if gaps else 0.0

def brier_score_multiclass(probs: np.ndarray, labels: np.ndarray, num_classes: int = NUM_CLASSES) -> float:
    onehot = np.eye(num_classes)[labels]
    return float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))

def compute_class_weights(samples: List[Dict], num_classes: int = NUM_CLASSES) -> torch.Tensor:
    counts = np.zeros(num_classes)
    for s in samples:
        counts[s["label"]] += 1
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


class CAWLPlus(nn.Module):
    def __init__(self, class_weights: torch.Tensor, num_classes: int = NUM_CLASSES,
                 emb_dim: int = 128, label_smoothing: float = 0.05,
                 target_confidence_cap: float = 0.85, lambda_max: float = 0.3,
                 anneal_epochs: int = 15, proto_momentum: float = 0.9,
                 proto_margin: float = 0.5, beta: float = 0.1,
                 proto_warmup_epochs: int = 8):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.label_smoothing = label_smoothing
        self.target_confidence_cap = target_confidence_cap
        self.lambda_max = lambda_max
        self.anneal_epochs = anneal_epochs
        self.current_epoch = 0

        self.num_classes = num_classes
        self.proto_momentum = proto_momentum
        self.proto_margin = proto_margin
        self.beta = beta
        self.proto_warmup_epochs = proto_warmup_epochs
        self.register_buffer("prototypes", torch.zeros(num_classes, emb_dim))
        self.register_buffer("proto_initialized", torch.zeros(num_classes, dtype=torch.bool))

    def _lambda_t(self):
        frac = min(self.current_epoch / max(self.anneal_epochs, 1), 1.0)
        return self.lambda_max * frac

    @torch.no_grad()
    def _update_prototypes(self, emb: torch.Tensor, targets: torch.Tensor):
        for c in range(self.num_classes):
            mask = targets == c
            if mask.sum() == 0:
                continue
            batch_mean = emb[mask].mean(dim=0)
            if not self.proto_initialized[c]:
                self.prototypes[c] = batch_mean
                self.proto_initialized[c] = True
            else:
                self.prototypes[c] = (self.proto_momentum * self.prototypes[c] +
                                       (1 - self.proto_momentum) * batch_mean)

    def _prototype_margin_loss(self, emb: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if not self.proto_initialized.all():
            return torch.tensor(0.0, device=emb.device)
        protos = self.prototypes.to(emb.device)
        d = torch.cdist(emb, protos) ** 2
        d_pos = d.gather(1, targets.view(-1, 1)).squeeze(1)
        d_masked = d.clone()
        d_masked.scatter_(1, targets.view(-1, 1), float("inf"))
        d_neg = d_masked.min(dim=1).values
        return F.relu(d_pos - d_neg + self.proto_margin).mean()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor,
                emb: Optional[torch.Tensor] = None) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.class_weights,
                              label_smoothing=self.label_smoothing)
        probs = F.softmax(logits, dim=1)
        conf, _ = probs.max(dim=1)
        pred = logits.argmax(dim=1)
        correct = (pred == targets).float()
        overconf = F.relu(conf - self.target_confidence_cap) * correct
        conf_penalty = overconf.mean()
        loss = ce + self._lambda_t() * conf_penalty

        if emb is not None and self.training:
            self._update_prototypes(emb.detach(), targets)
        if emb is not None and self.current_epoch >= self.proto_warmup_epochs:
            loss = loss + self.beta * self._prototype_margin_loss(emb, targets)
        return loss

def make_criterion(class_weights: torch.Tensor) -> "CAWLPlus":
    """Always use this instead of calling CAWLPlus(...) directly -- .to(DEVICE)
    on the module moves every registered buffer/parameter it owns. emb_dim is
    pinned explicitly to SMGF_MSCAECA_CONFIG['head_hidden'] (128) rather than
    relying on CAWLPlus's default, so this can never silently drift out of
    sync with the model's SmallHead again if either config changes later."""
    return CAWLPlus(class_weights, emb_dim=SMGF_MSCAECA_CONFIG['head_hidden']).to(DEVICE)

print("CAWLPlus ready (emb_dim pinned to SmallHead's head_hidden=128 for SMGF-Net + MS-CA-ECA).")


## §9 · Dataset Scanning & Leakage-Check Utilities — reused verbatim

In [ ]:
def _file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def dedupe_splits(splits: Dict[str, List[Dict]],
                   priority: List[str] = ["test", "val", "train"]) -> Dict[str, List[Dict]]:
    hash_to_owner: Dict[str, Tuple[str, int]] = {}
    to_remove: Dict[str, set] = {name: set() for name in splits}
    n_cross = n_intra = 0
    for name in sorted(splits, key=lambda n: priority.index(n) if n in priority else 99):
        for idx, s in enumerate(splits[name]):
            h = _file_hash(s["path"])
            if h in hash_to_owner:
                owner_name, owner_idx = hash_to_owner[h]
                if owner_name == name:
                    n_intra += 1
                else:
                    n_cross += 1
                to_remove[name].add(idx)
            else:
                hash_to_owner[h] = (name, idx)
    cleaned = {name: [s for i, s in enumerate(samples) if i not in to_remove[name]]
               for name, samples in splits.items()}
    print(f"[LEAKAGE CHECK] {n_cross} cross-split leak(s) + {n_intra} intra-split "
          f"duplicate(s) auto-removed ({n_cross + n_intra} total).")
    for name in splits:
        removed = len(to_remove[name])
        if removed:
            print(f"  {name}: {len(splits[name])} -> {len(cleaned[name])} ({removed} removed)")
    return cleaned

def scan_dataset_generic(root: Path, folder_to_class: Dict[str, str], split_name: str) -> List[Dict[str, Any]]:
    samples = []
    for folder_name, cls in folder_to_class.items():
        cls_dir = root / folder_name
        if not cls_dir.exists():
            print(f"  [WARN] missing: {cls_dir}")
            continue
        n_before = len(samples)
        for p in cls_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                samples.append({"path": str(p), "label": CLASS_TO_IDX[cls],
                                 "class_name": cls, "split": split_name})
        print(f"  {split_name}/{folder_name} -> {cls}: {len(samples) - n_before} images")
    print(f"{split_name}: {len(samples)} images across {NUM_CLASSES} classes")
    return samples

print("Scanner + leakage-check utilities ready.")


## §10 · Dataset Paths — BRISC2025

In [ ]:
BRISC_ROOT = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task")
BRISC_TRAIN_ROOT = BRISC_ROOT / "train"
BRISC_TEST_ROOT  = BRISC_ROOT / "test"
BRISC_FOLDER_TO_CLASS = {
    "glioma": "glioma", "meningioma": "meningioma",
    "no_tumor": "notumor", "pituitary": "pituitary",
}
print(f"BRISC2025 train exists: {BRISC_TRAIN_ROOT.exists()} -> {BRISC_TRAIN_ROOT}")
print(f"BRISC2025 test  exists: {BRISC_TEST_ROOT.exists()} -> {BRISC_TEST_ROOT}")


In [ ]:
print("BRISC_ROOT exists:", BRISC_ROOT.exists(), "->", BRISC_ROOT)
if BRISC_ROOT.exists():
    print("Contents of BRISC_ROOT:", list(BRISC_ROOT.iterdir()))
else:
    print("Fix: open the Kaggle 'Add Input' panel, find your BRISC2025 dataset's real path, "
          "and update BRISC_ROOT above to match it exactly.")


## §11 · Local Client Training & Evaluation Utilities

`local_train_epoch` is the shared local optimizer step for both
strategies. A FedProx-style proximal term is optional (`mu > 0`) so the
*same* function serves FedAvg (mu=0) and FedProx (mu=FEDPROX_MU).

In [ ]:
def local_train_epoch(model: nn.Module, loader: DataLoader, optimizer, criterion: CAWLPlus,
                       global_state: Optional[Dict[str, torch.Tensor]] = None, mu: float = 0.0,
                       scaler: Optional[GradScaler] = None) -> Tuple[float, float]:
    """One local epoch. If mu > 0 and global_state is given, adds the FedProx
    proximal term (mu/2)||w - w_global||^2 over all *trainable* parameters."""
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        def _prox_term():
            if mu <= 0 or global_state is None:
                return 0.0
            term = 0.0
            for name, p in model.named_parameters():
                if name in global_state:
                    term = term + ((p - global_state[name].to(p.device)) ** 2).sum()
            return 0.5 * mu * term

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"]) + _prox_term()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"]) + _prox_term()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def local_evaluate(model: nn.Module, loader: DataLoader, criterion: CAWLPlus) -> Dict[str, Any]:
    model.eval()
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        out   = model(img)
        loss  = criterion(out["logits"], label, out["emb"])
        total_loss += loss.item() * label.size(0)
        probs = torch.softmax(out["logits"], dim=1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(out["logits"].argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    n = max(len(all_labels), 1)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc_val = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc_val = float("nan")
    ece, bin_accs, bin_confs, bin_counts = expected_calibration_error(y_prob, y_true)
    mce = maximum_calibration_error(bin_accs, bin_confs, bin_counts)
    brier = brier_score_multiclass(y_prob, y_true) if n > 0 else float("nan")

    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred) if n else 0.0,
        "balanced_acc": balanced_accuracy_score(y_true, y_pred) if n else 0.0,
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auc": auc_val,
        "mcc": matthews_corrcoef(y_true, y_pred) if n else 0.0,
        "kappa": cohen_kappa_score(y_true, y_pred) if n else 0.0,
        "ece": ece, "mce": mce, "brier": brier,
        "n": n,
    }

print("Local client training/evaluation utilities ready.")


## §12 · Client Simulation — Non-IID split, Dirichlet(α) label skew (Scenario B)

In [ ]:
def load_brisc_pool() -> List[Dict]:
    train_samples = scan_dataset_generic(BRISC_TRAIN_ROOT, BRISC_FOLDER_TO_CLASS, "train")
    test_samples  = scan_dataset_generic(BRISC_TEST_ROOT, BRISC_FOLDER_TO_CLASS, "test")
    cleaned = dedupe_splits({"train": train_samples, "test": test_samples})
    return cleaned["train"], cleaned["test"]

# BRISC_TRAIN_POOL, GLOBAL_TEST_SAMPLES = load_brisc_pool()   # uncomment on Kaggle


### Non-IID split: label-distribution skew via Dirichlet(α) allocation (Hsu et al., 2019)

In [ ]:
def split_scenario_b_noniid(pool: List[Dict], n_clients: int = NUM_CLIENTS,
                             alpha: float = NONIID_ALPHA, seed: int = SEED,
                             min_samples_per_client: int = 20) -> List[List[Dict]]:
    """Non-IID label-skew split via a Dirichlet(alpha) allocation (Hsu et al.,
    2019, "Measuring the Effects of Non-Identical Data Distribution for
    Federated Visual Classification"). For each class independently, draws a
    proportion vector p ~ Dir(alpha, ..., alpha) over the n_clients clients
    and hands that class's samples out according to p. Smaller alpha ->
    each client's local label distribution concentrates on just one or two
    classes (severe skew, e.g. one "hospital" that only ever sees glioma
    scans); alpha -> infinity recovers the old IID split. Re-draws the
    Dirichlet allocation (retrying with a fresh RNG state) if any client
    would end up with too few samples to train on, so a single unlucky draw
    can't leave a client's local loader empty."""
    rng = np.random.RandomState(seed)
    by_class: Dict[int, List[Dict]] = {c: [] for c in range(NUM_CLASSES)}
    for s in pool:
        by_class[s["label"]].append(s)

    for attempt in range(50):
        clients: List[List[Dict]] = [[] for _ in range(n_clients)]
        for c, items in by_class.items():
            idx = rng.permutation(len(items))
            proportions = rng.dirichlet(alpha=np.full(n_clients, alpha))
            # convert proportions -> cut points over the shuffled index array
            cuts = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
            splits = np.split(idx, cuts)
            for k, chunk in enumerate(splits):
                clients[k].extend([items[i] for i in chunk])
        sizes = [len(c) for c in clients]
        if min(sizes) >= min_samples_per_client:
            break
    else:
        raise RuntimeError(
            f"Could not reach min_samples_per_client={min_samples_per_client} per "
            f"client after 50 Dirichlet re-draws at alpha={alpha}; try a larger "
            f"alpha or a lower min_samples_per_client.")

    for k in range(n_clients):
        rng.shuffle(clients[k])
    return clients


### Client distribution visualization

In [ ]:
if "BRISC_TRAIN_POOL" not in globals() or len(BRISC_TRAIN_POOL) == 0:
    print("BRISC_TRAIN_POOL is missing or empty -- loading it now.")
    BRISC_TRAIN_POOL, GLOBAL_TEST_SAMPLES = load_brisc_pool()
    print(f"Loaded: train_pool={len(BRISC_TRAIN_POOL)}  global_test={len(GLOBAL_TEST_SAMPLES)}")

assert len(BRISC_TRAIN_POOL) > 0, (
    "BRISC_TRAIN_POOL is empty -- BRISC_ROOT almost certainly points at the wrong path. "
    "Check BRISC_ROOT.exists() (see the diagnostic cell in \u00a7\u00a710) and fix BRISC_ROOT if needed."
)

def plot_client_distribution(clients: List[List[Dict]], title: str,
                              client_labels: Optional[List[str]] = None,
                              save_path: Optional[str] = None):
    n_clients = len(clients)
    client_labels = client_labels or [f"Client {i+1}" for i in range(n_clients)]
    counts = np.zeros((n_clients, NUM_CLASSES))
    for k, c in enumerate(clients):
        for s in c:
            counts[k, s["label"]] += 1
    fig, ax = plt.subplots(figsize=(8, 5))
    bottom = np.zeros(n_clients)
    colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))
    for ci, cname in enumerate(CLASS_NAMES):
        ax.bar(client_labels, counts[:, ci], bottom=bottom, label=cname, color=colors[ci])
        bottom += counts[:, ci]
    ax.set_ylabel("# samples"); ax.set_title(title)
    ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return counts

clients_b = split_scenario_b_noniid(BRISC_TRAIN_POOL, alpha=NONIID_ALPHA)
plot_client_distribution(clients_b, f"Non-IID Client Distribution (Dirichlet \u03b1={NONIID_ALPHA})",
                          save_path=str(OUT_DIR / "fig_client_distribution.png"))
client_sizes = [len(c) for c in clients_b]
print("Per-client sample counts:", client_sizes)


## §13 · Aggregation Strategies (server-side)

One shared `federated_aggregate` function parameterized by `strategy`, so both
methods reuse identical client-selection, local-training, and evaluation
code -- the only thing that changes is the proximal term `mu` used during
local training (FedProx). The `exclude_bn` plumbing in
`get_trainable_state` is retained but now always False, since FedBN has
been dropped; it is left in place rather than rewriting the orchestrator.

In [ ]:
def is_bn_param(name: str) -> bool:
    return any(k in name.lower() for k in BN_LAYER_KEYWORDS)

def get_trainable_state(model: nn.Module, exclude_bn: bool) -> Dict[str, torch.Tensor]:
    sd = model.state_dict()
    if not exclude_bn:
        return {k: v.clone() for k, v in sd.items()}
    return {k: v.clone() for k, v in sd.items() if not is_bn_param(k)}

def sample_count_weights(client_n: List[int]) -> np.ndarray:
    n = np.array(client_n, dtype=np.float64)
    return n / n.sum()

def weighted_average_state_dicts(state_dicts: List[Dict[str, torch.Tensor]],
                                  alphas: np.ndarray) -> Dict[str, torch.Tensor]:
    keys = state_dicts[0].keys()
    avg = {}
    for k in keys:
        stacked = torch.stack([sd[k].float() * a for sd, a in zip(state_dicts, alphas)], dim=0)
        avg[k] = stacked.sum(dim=0).to(state_dicts[0][k].dtype)
    return avg

STRATEGY_CONFIG = {
    "fedavg":  {"exclude_bn": False, "mu": 0.0},
    "fedprox": {"exclude_bn": False, "mu": FEDPROX_MU},
}

def federated_aggregate(strategy: str, client_states: List[Dict[str, torch.Tensor]],
                         client_n: List[int]) -> Tuple[Dict[str, torch.Tensor], np.ndarray]:
    alphas = sample_count_weights(client_n)
    return weighted_average_state_dicts(client_states, alphas), alphas

print("Aggregation strategies ready:", list(STRATEGY_CONFIG.keys()))


## §14 · Federated Training Orchestrator — `run_federated_experiment()`

Runs `COMM_ROUNDS` rounds of: (1) each client loads the current global
state into its local model, (2) local
training for `LOCAL_EPOCHS` (with the FedProx proximal term when `mu>0`),
(3) server aggregates by sample-count weighting, (4) evaluate the new global
model on the held-out global test set every round for the convergence
curves used in §Visualizations.

In [ ]:
def make_client_loaders(client_samples: List[Dict], val_frac: float = 0.15, seed: int = SEED
                         ) -> Tuple[DataLoader, DataLoader]:
    labels = [s["label"] for s in client_samples]
    label_counts = {c: labels.count(c) for c in set(labels)}
    can_stratify = len(client_samples) >= 4 and min(label_counts.values()) >= 2

    if len(client_samples) < 4:
        tr, va = client_samples, client_samples
    elif can_stratify:
        tr, va = train_test_split(client_samples, test_size=val_frac, random_state=seed,
                                   stratify=labels)
    else:
        tr, va = train_test_split(client_samples, test_size=val_frac, random_state=seed,
                                   stratify=None)

    tr_ds = BrainTumorDataset(tr, is_train=True, augmenter=train_augment)
    va_ds = BrainTumorDataset(va, is_train=False)
    return (DataLoader(tr_ds, **_loader_kwargs(shuffle=True)),
            DataLoader(va_ds, **_loader_kwargs(shuffle=False)))


def run_federated_experiment(strategy: str, client_partitions: List[List[Dict]],
                              global_test_samples: List[Dict],
                              comm_rounds: int = COMM_ROUNDS, local_epochs: int = LOCAL_EPOCHS,
                              seed: int = SEED, verbose_every: int = 5,
                              mu_override: Optional[float] = None,
                              model_builder=None, criterion_factory=None,
                              use_amp: Optional[bool] = None,
                              tag: Optional[str] = None) -> Dict[str, Any]:
    """Runs one strategy end-to-end across `comm_rounds`. Returns history +
    final global model + per-round per-client metrics.

    Optional arguments (defaults reproduce the original behaviour):
      mu_override       FedProx mu for this run only (mu sweep).
      model_builder     callable returning a fresh model (non-private GroupNorm control).
      criterion_factory callable(class_weights) -> criterion (default make_criterion).
      use_amp           override USE_AMP (the DP arm runs without AMP).
      tag               label stored in the result.

    The global test set is evaluated after every round for the curves only;
    nothing is selected with it and the reported metrics are the last round's.
    """
    cfg = dict(STRATEGY_CONFIG[strategy])
    if mu_override is not None:
        cfg["mu"] = float(mu_override)
    builder      = model_builder or build_smgf_mscaeca
    crit_factory = criterion_factory or make_criterion
    amp_on       = USE_AMP if use_amp is None else use_amp
    tag          = tag or strategy
    n_clients    = len(client_partitions)

    # Seed everything here so FedAvg and FedProx start from identical weights
    # and a run does not depend on how many RNG draws earlier runs consumed.
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    global_model = builder().to(DEVICE)
    class_weights = compute_class_weights([s for c in client_partitions for s in c])  # pooled over all clients

    client_loaders = [make_client_loaders(c, seed=seed) for c in client_partitions]
    client_n = [len(c) for c in client_partitions]
    client_train_n = [len(tl.dataset) for tl, _ in client_loaders]
    test_loader = DataLoader(BrainTumorDataset(global_test_samples, is_train=False),
                              **_loader_kwargs(shuffle=False))

    client_bn_state: Dict[int, Dict[str, torch.Tensor]] = {}
    history = {"round": [], "acc": [], "auc": [], "ece": [], "mce": [], "brier": [],
               "f1_mac": [], "balanced_acc": [], "comm_bytes_round": []}
    client_history: Dict[int, List[Dict]] = {k: [] for k in range(n_clients)}

    scaler = GradScaler() if amp_on and DEVICE == "cuda" else None
    param_bytes = sum(p.numel() for p in global_model.parameters()) * 4  # fp32

    for rnd in range(1, comm_rounds + 1):
        client_states = []
        for k in range(n_clients):
            local_model = copy.deepcopy(global_model)
            if cfg["exclude_bn"]:
                if k not in client_bn_state:
                    client_bn_state[k] = {kk: vv.clone() for kk, vv in global_model.state_dict().items()
                                          if is_bn_param(kk)}
                local_sd = local_model.state_dict()
                for key, val in client_bn_state[k].items():
                    local_sd[key] = val.clone()
                local_model.load_state_dict(local_sd)

            criterion = crit_factory(class_weights)
            criterion.current_epoch = rnd
            # The optimizer is re-created every round: AdamW moments do not carry over.
            optimizer = torch.optim.AdamW([p for p in local_model.parameters() if p.requires_grad],
                                           lr=CLIENT_LR, weight_decay=CLIENT_WEIGHT_DECAY)
            train_loader, val_loader = client_loaders[k]
            gstate_for_prox = {n: p.clone() for n, p in global_model.named_parameters()} if cfg["mu"] > 0 else None
            for _ in range(local_epochs):
                local_train_epoch(local_model, train_loader, optimizer, criterion,
                                   global_state=gstate_for_prox, mu=cfg["mu"], scaler=scaler)

            if cfg["exclude_bn"]:
                client_bn_state[k] = {kk: vv.clone() for kk, vv in local_model.state_dict().items()
                                      if is_bn_param(kk)}
            client_states.append(get_trainable_state(local_model, exclude_bn=cfg["exclude_bn"]))
            m = local_evaluate(local_model, val_loader, criterion)
            client_history[k].append({"round": rnd, "balanced_acc": m["balanced_acc"],
                                       "f1_mac": m["f1_mac"]})
            del local_model

        agg_state, alphas = federated_aggregate(strategy, client_states, client_n)
        gsd = global_model.state_dict()
        gsd.update(agg_state)
        global_model.load_state_dict(gsd)

        eval_criterion = crit_factory(class_weights); eval_criterion.current_epoch = rnd
        m = local_evaluate(global_model, test_loader, eval_criterion)
        history["round"].append(rnd)
        history["acc"].append(m["acc"]); history["auc"].append(m["auc"])
        history["ece"].append(m["ece"]); history["mce"].append(m["mce"]); history["brier"].append(m["brier"])
        history["f1_mac"].append(m["f1_mac"]); history["balanced_acc"].append(m["balanced_acc"])
        payload = param_bytes * (0.996 if cfg["exclude_bn"] else 1.0)
        history["comm_bytes_round"].append(2 * n_clients * payload)

        if rnd % verbose_every == 0 or rnd == 1:
            print(f"[{tag}] round {rnd:03d}  acc={m['acc']:.4f}  auc={m['auc']:.4f}  "
                  f"ece={m['ece']:.4f}")

    return {"strategy": strategy, "tag": tag, "seed": seed, "mu": cfg["mu"],
            "model": global_model, "history": history,
            "client_history": client_history, "final_metrics": m,
            "client_n": client_n, "client_train_n": client_train_n}

print("run_federated_experiment() ready for strategies:", list(STRATEGY_CONFIG.keys()))


## §14.1 · Persistence and heterogeneity helpers

Self-contained on purpose (they do not need anything from §16–§23), so they also work in a DP-only session.
`persist_run()` writes the artifacts that the per-class tables, confusion matrices and later evaluation-only
analyses need, so none of them requires retraining.


In [ ]:
import json
from scipy.spatial.distance import jensenshannon

@torch.no_grad()
def _collect_preds(model: nn.Module, samples: List[Dict]) -> Dict[str, np.ndarray]:
    model.eval()
    loader = DataLoader(BrainTumorDataset(samples, is_train=False), **_loader_kwargs(shuffle=False))
    probs, preds, labels, paths = [], [], [], []
    for batch in loader:
        out = model(batch["image"].to(DEVICE, non_blocking=True))
        probs.append(torch.softmax(out["logits"], dim=1).cpu().numpy())
        preds.append(out["logits"].argmax(1).cpu().numpy())
        labels.append(batch["label"].numpy())
        paths.extend(batch["path"])
    return {"y_prob": np.concatenate(probs), "y_pred": np.concatenate(preds),
            "y_true": np.concatenate(labels), "paths": np.array(paths)}

def per_class_table(y_true, y_pred) -> pd.DataFrame:
    rep = classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                                target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    return pd.DataFrame({c: rep[c] for c in CLASS_NAMES}).T[["precision", "recall", "f1-score", "support"]]

def tumor_vs_notumor(y_true, y_pred) -> Dict[str, float]:
    """Tumor-detection view of the four-class output: any of the three tumor classes
    predicted as no-tumor is a missed tumor, whichever tumor class it was."""
    nt = CLASS_TO_IDX["notumor"]
    is_tumor = (y_true != nt)
    return {"n_tumor": int(is_tumor.sum()),
            "tumor_missed_as_notumor": int((y_pred[is_tumor] == nt).sum()),
            "tumor_detection_sensitivity": float((y_pred[is_tumor] != nt).mean()),
            "notumor_recall": float((y_pred[~is_tumor] == nt).mean())}

def persist_run(res: Dict[str, Any], name: str, samples: Optional[List[Dict]] = None) -> Optional[Dict[str, Any]]:
    """Save history, weights, test predictions, per-class table and confusion matrix of one run."""
    model = res.get("model")
    if model is None:
        print(f"[persist] {name}: no model in memory (replayed run) -- nothing to save")
        return None
    samples = GLOBAL_TEST_SAMPLES if samples is None else samples
    p = _collect_preds(model, samples)
    np.savez_compressed(OUT_DIR / f"{name}_test_predictions.npz", **p)
    torch.save(model.state_dict(), OUT_DIR / f"{name}_global_model.pt")
    if res.get("history"):
        pd.DataFrame(res["history"]).to_csv(OUT_DIR / f"{name}_history.csv", index=False)
    pc = per_class_table(p["y_true"], p["y_pred"])
    pc.to_csv(OUT_DIR / f"{name}_per_class.csv")
    cm = confusion_matrix(p["y_true"], p["y_pred"], labels=list(range(NUM_CLASSES)))
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUT_DIR / f"{name}_confusion.csv")
    row = {"run": name, "acc": float((p["y_true"] == p["y_pred"]).mean()), "n": int(len(p["y_true"]))}
    row.update({f"recall_{c}": float(pc.loc[c, "recall"]) for c in CLASS_NAMES})
    row.update({f"precision_{c}": float(pc.loc[c, "precision"]) for c in CLASS_NAMES})
    row.update(tumor_vs_notumor(p["y_true"], p["y_pred"]))
    print(f"[persist] {name}: acc={row['acc']:.4f}  tumor-detection sensitivity="
          f"{row['tumor_detection_sensitivity']:.4f}  saved to {OUT_DIR}")
    return row

def partition_heterogeneity(clients: List[List[Dict]], name: str) -> Dict[str, Any]:
    """Label-distribution heterogeneity of a client partition.

    JSD : Jensen-Shannon divergence in bits (base 2, range [0, 1]); the JS *distance*
          (its square root) is stored as well, because papers report either.
    EMD : earth mover's distance between label distributions with a 0-1 ground metric
          between classes (classes are not ordinal), which equals total variation, 0.5*L1.
    Also stores client sizes before and after the 15 % local hold-out.
    """
    counts = np.zeros((len(clients), NUM_CLASSES))
    for k, c in enumerate(clients):
        for s in c:
            counts[k, s["label"]] += 1
    P = counts / counts.sum(axis=1, keepdims=True)
    g = counts.sum(axis=0) / counts.sum()
    K = len(clients)
    jsd = np.array([[jensenshannon(P[i], P[j], base=2) ** 2 for j in range(K)] for i in range(K)])
    emd = np.array([[0.5 * np.abs(P[i] - P[j]).sum() for j in range(K)] for i in range(K)])
    iu = np.triu_indices(K, 1)
    sizes_total = [len(c) for c in clients]
    sizes_train = []
    for c in clients:
        tl, vl = make_client_loaders(c)
        sizes_train.append((len(tl.dataset), len(vl.dataset)))
    out = {
        "name": name, "n_clients": K, "client_total": sizes_total,
        "client_train_after_15pct_holdout": [a for a, _ in sizes_train],
        "client_local_val": [b for _, b in sizes_train],
        "class_counts": counts.astype(int).tolist(),
        "jsd_pairwise_max": float(jsd[iu].max()), "jsd_pairwise_mean": float(jsd[iu].mean()),
        "js_distance_pairwise_max": float(np.sqrt(jsd[iu].max())),
        "jsd_to_global_max": float(max(jensenshannon(P[i], g, base=2) ** 2 for i in range(K))),
        "emd_pairwise_max": float(emd[iu].max()), "emd_pairwise_mean": float(emd[iu].mean()),
    }
    with open(OUT_DIR / f"partition_heterogeneity_{name}.json", "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{name}] sizes={sizes_total}  train after hold-out={out['client_train_after_15pct_holdout']}")
    print(f"[{name}] JSD  max/mean = {out['jsd_pairwise_max']:.4f} / {out['jsd_pairwise_mean']:.4f}"
          f"   (JS distance max {out['js_distance_pairwise_max']:.4f})")
    print(f"[{name}] EMD  max/mean = {out['emd_pairwise_max']:.4f} / {out['emd_pairwise_mean']:.4f}")
    return out

print("Persistence and heterogeneity helpers ready.")


## §15 · Run the Comparison — FedAvg vs. FedProx (Non-IID)

Trains all three strategies once each (single seed, Dirichlet(α) non-IID
client split) and keeps every trained global model in memory so §16-21 below can run test-set
diagnostics (ROC/PR, confusion matrix, t-SNE, Grad-CAM++, Score-CAM, LIME)
directly on them.

In [ ]:
if "BRISC_TRAIN_POOL" not in globals() or len(BRISC_TRAIN_POOL) == 0:
    print("BRISC_TRAIN_POOL is missing or empty -- loading it now.")
    BRISC_TRAIN_POOL, GLOBAL_TEST_SAMPLES = load_brisc_pool()
    print(f"Loaded: train_pool={len(BRISC_TRAIN_POOL)}  global_test={len(GLOBAL_TEST_SAMPLES)}")

assert len(BRISC_TRAIN_POOL) > 0, (
    "BRISC_TRAIN_POOL is empty -- BRISC_ROOT almost certainly points at the wrong path. "
    "Check BRISC_ROOT.exists() and BRISC_ROOT in the §§10 cell against your attached Kaggle dataset."
)
assert FL_SEEDS[0] == SEED, "the first FL seed must equal SEED: it reproduces the paper partition the DP arm reuses"

STRATEGIES = ["fedavg", "fedprox"]
print(f"client AdamW: lr={CLIENT_LR:g} wd={CLIENT_WEIGHT_DECAY:g}  rounds={COMM_ROUNDS}  "
      f"local_epochs={LOCAL_EPOCHS}  seeds={FL_SEEDS}")

fed_results: Dict[str, Dict[str, Any]] = {}          # first seed only (what the DP arm compares against)
fed_results_by_seed: Dict[Tuple[str, int], Dict[str, Any]] = {}
partitions_by_seed: Dict[int, List[List[Dict]]] = {}
rows, rows_all = [], []
for sd in FL_SEEDS:
    # seed -> Dirichlet draw. Seed 42 is the partition used in the paper.
    clients_sd = split_scenario_b_noniid(BRISC_TRAIN_POOL, alpha=NONIID_ALPHA, seed=sd)
    partitions_by_seed[sd] = clients_sd
    print(f"seed {sd}: client sizes {[len(c) for c in clients_sd]}")
    if sd == FL_SEEDS[0]:
        clients_noniid = clients_sd
    for strategy in STRATEGIES:
        print(f"\n=== seed={sd}  strategy={strategy} ===")
        res = run_federated_experiment(strategy, clients_sd, GLOBAL_TEST_SAMPLES, seed=sd,
                                       tag=f"{strategy}_seed{sd}")
        fed_results_by_seed[(strategy, sd)] = res
        fm = res["final_metrics"]
        row = {"strategy": strategy, "acc": fm["acc"], "balanced_acc": fm["balanced_acc"],
               "precision": fm["precision_mac"], "recall": fm["recall_mac"],
               "f1": fm["f1_mac"], "auc": fm["auc"], "mcc": fm["mcc"],
               "ece": fm["ece"], "mce": fm["mce"], "brier": fm["brier"]}
        rows_all.append({**row, "seed": sd})
        if sd == FL_SEEDS[0]:
            fed_results[strategy] = res
            rows.append(row)

results_df = pd.DataFrame(rows)
# File name kept: the DP-only replay flow (§23.9) reads exactly this CSV.
results_df.to_csv(OUT_DIR / "fedavg_fedprox_fedbn_comparison.csv", index=False)
by_seed_df = pd.DataFrame(rows_all)
by_seed_df.to_csv(OUT_DIR / "fedavg_fedprox_by_seed.csv", index=False)
print("\n", results_df)

if len(FL_SEEDS) > 1:
    from scipy import stats as _st
    print("\nAcross seeds (each seed = new Dirichlet draw + new init); mean +/- SD, 95% t-interval:")
    for strategy in STRATEGIES:
        a = by_seed_df.loc[by_seed_df.strategy == strategy, "acc"].to_numpy()
        h = _st.t.ppf(0.975, len(a) - 1) * a.std(ddof=1) / np.sqrt(len(a))
        print(f"  {strategy:8s} acc = {a.mean():.4f} +/- {a.std(ddof=1):.4f}   "
              f"[{a.mean()-h:.4f}, {a.mean()+h:.4f}]   per seed: {np.round(a, 4).tolist()}")
    d = (by_seed_df[by_seed_df.strategy == "fedprox"].set_index("seed")["acc"]
         - by_seed_df[by_seed_df.strategy == "fedavg"].set_index("seed")["acc"])
    print(f"  FedProx - FedAvg per seed: {np.round(d.to_numpy(), 4).tolist()}")

trained_models = {s: fed_results[s]["model"] for s in STRATEGIES}
histories = {s: fed_results[s]["history"] for s in STRATEGIES}
client_histories = {s: fed_results[s]["client_history"] for s in STRATEGIES}


## §15.0 · Persist every baseline run

Saves history, weights, test predictions, per-class precision/recall and the confusion matrix for each
(strategy, seed). Everything that needs predictions later (per-class tables, tumor-vs-no-tumor errors,
McNemar, calibration) can then be computed without retraining.


In [ ]:
persist_rows = []
for (strategy, sd), res in fed_results_by_seed.items():
    r = persist_run(res, f"fl_{strategy}_seed{sd}")
    if r is not None:
        persist_rows.append(r)
if persist_rows:
    pd.DataFrame(persist_rows).to_csv(OUT_DIR / "fl_run_summary.csv", index=False)
    print(pd.DataFrame(persist_rows)[["run", "acc", "tumor_detection_sensitivity"]
                                     + [f"recall_{c}" for c in CLASS_NAMES]].to_string(index=False,
          float_format=lambda v: f"{v:.4f}"))
json.dump({"client_n": fed_results["fedavg"]["client_n"],
           "client_train_n_after_holdout": fed_results["fedavg"]["client_train_n"]},
          open(OUT_DIR / "fl_client_sizes.json", "w"), indent=2)


## §15.1 · FedProx mu sweep (optional)

Off by default (`RUN_MU_SWEEP = False`). Uses the first FL seed and the same Dirichlet partition as §15, so the rows are
directly comparable with the FedProx row above. Per-round histories are saved for the convergence figure.


In [ ]:
mu_sweep = {}
if RUN_MU_SWEEP:
    for mu in FEDPROX_MU_GRID:
        print(f"\n=== FedProx mu={mu:g} (seed {FL_SEEDS[0]}) ===")
        res = run_federated_experiment("fedprox", clients_noniid, GLOBAL_TEST_SAMPLES, seed=FL_SEEDS[0],
                                       mu_override=mu, tag=f"fedprox_mu{mu:g}")
        mu_sweep[mu] = res
        pd.DataFrame(res["history"]).to_csv(OUT_DIR / f"fl_fedprox_mu{mu:g}_history.csv", index=False)
    mu_df = pd.DataFrame([{"mu": mu, **{k: r["final_metrics"][k] for k in ["acc", "balanced_acc", "f1_mac", "auc", "ece"]}}
                          for mu, r in mu_sweep.items()])
    mu_df.to_csv(OUT_DIR / "fl_fedprox_mu_sweep.csv", index=False)
    print(mu_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
else:
    print("RUN_MU_SWEEP = False -- skipping the mu sweep.")


## §16 · Visualizations — convergence, communication cost, calibration

In [ ]:
def plot_metric_vs_rounds(histories: Dict[str, Dict[str, list]], metric: str = "acc",
                           title: Optional[str] = None, save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(8, 5))
    for strategy, h in histories.items():
        ax.plot(h["round"], h[metric], label=strategy, linewidth=2)
    ax.set_xlabel("Communication round"); ax.set_ylabel(metric)
    ax.set_title(title or f"Global {metric} vs. communication rounds")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def plot_communication_cost(histories: Dict[str, Dict[str, list]], save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(7, 5))
    totals = {s: np.sum(h.get("comm_bytes_round", [0])) / 1e6 for s, h in histories.items()}
    ax.bar(list(totals.keys()), list(totals.values()), color=plt.cm.Set2(np.linspace(0, 1, len(totals))))
    ax.set_ylabel("Total communication (MB)"); ax.set_title("Total Communication Cost by Strategy")
    plt.xticks(rotation=20); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def plot_strategy_comparison_bars(results_df: pd.DataFrame, metric: str = "acc",
                                   save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(results_df["strategy"], results_df[metric], color=plt.cm.Set2(np.linspace(0, 1, len(results_df))))
    ax.set_ylabel(metric); ax.set_title(f"{metric} — FedAvg vs FedProx")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def plot_reliability_diagram(probs: np.ndarray, labels: np.ndarray, title: str = "Reliability Diagram",
                              n_bins: int = 15, save_path: Optional[str] = None) -> Dict[str, float]:
    ece, bin_accs, bin_confs, bin_counts = expected_calibration_error(probs, labels, n_bins)
    mce = maximum_calibration_error(bin_accs, bin_confs, bin_counts)
    brier = brier_score_multiclass(probs, labels)
    centers = (np.linspace(0, 1, n_bins + 1)[:-1] + np.linspace(0, 1, n_bins + 1)[1:]) / 2
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.bar(centers, bin_accs, width=1/n_bins, edgecolor="black", alpha=0.7, label="Accuracy")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(f"{title}\nECE={ece:.4f}  MCE={mce:.4f}  Brier={brier:.4f}")
    ax.legend(); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return {"ece": ece, "mce": mce, "brier": brier}

plot_metric_vs_rounds(histories, metric="acc", save_path=str(OUT_DIR / "fig_acc_vs_rounds.png"))
plot_metric_vs_rounds(histories, metric="auc", save_path=str(OUT_DIR / "fig_auc_vs_rounds.png"))
plot_communication_cost(histories, save_path=str(OUT_DIR / "fig_comm_cost.png"))
plot_strategy_comparison_bars(results_df, metric="acc", save_path=str(OUT_DIR / "fig_acc_bars.png"))
print("Convergence / communication-cost / summary-bar plots ready.")


## §17 · Test-Set Diagnostics — predictions, classification report, confusion matrix

Runs every trained global model over the held-out `GLOBAL_TEST_SAMPLES`
once, caching predicted probabilities, hard predictions, and the 64-d
embedding (`emb`, from `ProjectionHead`) for every test image. Everything
below (ROC/PR curves, t-SNE, XAI) reuses this cache.

In [ ]:
@torch.no_grad()
def collect_test_predictions(model: nn.Module, samples: List[Dict]) -> Dict[str, np.ndarray]:
    model.eval()
    loader = DataLoader(BrainTumorDataset(samples, is_train=False), **_loader_kwargs(shuffle=False))
    all_probs, all_preds, all_labels, all_embs, all_paths = [], [], [], [], []
    for batch in loader:
        img = batch["image"].to(DEVICE, non_blocking=True)
        out = model(img)
        probs = torch.softmax(out["logits"], dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(out["logits"].argmax(1).cpu().numpy())
        all_labels.append(batch["label"].numpy())
        all_embs.append(out["emb"].cpu().numpy())
        all_paths.extend(batch["path"])
    return {"y_prob": np.concatenate(all_probs), "y_pred": np.concatenate(all_preds),
            "y_true": np.concatenate(all_labels), "emb": np.concatenate(all_embs),
            "paths": np.array(all_paths)}

test_predictions = {s: collect_test_predictions(trained_models[s], GLOBAL_TEST_SAMPLES) for s in STRATEGIES}

for s in STRATEGIES:
    p = test_predictions[s]
    print(f"\n===== {s.upper()} — Classification Report =====")
    print(classification_report(p["y_true"], p["y_pred"], target_names=CLASS_NAMES, digits=4))

def plot_confusion_matrix(y_true, y_pred, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

for s in STRATEGIES:
    p = test_predictions[s]
    plot_confusion_matrix(p["y_true"], p["y_pred"], f"Confusion Matrix — {s}",
                           save_path=str(OUT_DIR / f"fig_confusion_{s}.png"))


## §18 · ROC and Precision-Recall Curves (one-vs-rest, per class + macro)

In [ ]:
def plot_roc_curves(y_true, y_prob, title, save_path=None):
    fig, ax = plt.subplots(figsize=(6.5, 6))
    y_onehot = np.eye(NUM_CLASSES)[y_true]
    aucs = []
    for c in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_onehot[:, c], y_prob[:, c])
        roc_auc = auc(fpr, tpr); aucs.append(roc_auc)
        ax.plot(fpr, tpr, label=f"{CLASS_NAMES[c]} (AUC={roc_auc:.3f})")
    fpr_m, tpr_m, _ = roc_curve(y_onehot.ravel(), y_prob.ravel())
    ax.plot(fpr_m, tpr_m, "k--", linewidth=2, label=f"micro-avg (AUC={auc(fpr_m, tpr_m):.3f})")
    ax.plot([0, 1], [0, 1], color="gray", linestyle=":")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{title}\nmacro-AUC={np.mean(aucs):.4f}")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

def plot_pr_curves(y_true, y_prob, title, save_path=None):
    fig, ax = plt.subplots(figsize=(6.5, 6))
    y_onehot = np.eye(NUM_CLASSES)[y_true]
    aps = []
    for c in range(NUM_CLASSES):
        prec, rec, _ = precision_recall_curve(y_onehot[:, c], y_prob[:, c])
        ap = average_precision_score(y_onehot[:, c], y_prob[:, c]); aps.append(ap)
        ax.plot(rec, prec, label=f"{CLASS_NAMES[c]} (AP={ap:.3f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"{title}\nmacro-AP={np.mean(aps):.4f}")
    ax.legend(loc="lower left", fontsize=9)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

for s in STRATEGIES:
    p = test_predictions[s]
    plot_roc_curves(p["y_true"], p["y_prob"], f"ROC Curves — {s}",
                     save_path=str(OUT_DIR / f"fig_roc_{s}.png"))
    plot_pr_curves(p["y_true"], p["y_prob"], f"Precision-Recall Curves — {s}",
                   save_path=str(OUT_DIR / f"fig_pr_{s}.png"))


## §19 · t-SNE of the Embedding Space (64-d `emb` from `ProjectionHead`)

In [ ]:
def plot_tsne(emb, y_true, title, save_path=None, seed=SEED):
    n = emb.shape[0]
    perplexity = min(30, max(5, n // 20))
    tsne = TSNE(n_components=2, perplexity=perplexity, init="pca",
                learning_rate="auto", random_state=seed)
    proj = tsne.fit_transform(emb)
    fig, ax = plt.subplots(figsize=(7, 6))
    colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))
    for c in range(NUM_CLASSES):
        mask = y_true == c
        ax.scatter(proj[mask, 0], proj[mask, 1], s=12, alpha=0.7, color=colors[c], label=CLASS_NAMES[c])
    ax.set_title(title); ax.legend(title="Class"); ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

for s in STRATEGIES:
    p = test_predictions[s]
    plot_tsne(p["emb"], p["y_true"], f"t-SNE of test-set embeddings — {s}",
              save_path=str(OUT_DIR / f"fig_tsne_{s}.png"))


## §20 · Explainability — Grad-CAM++

Target layer: `model.multiscale.refine[0]`, the last depthwise-refinement
conv in the Spatial branch's multi-scale fusion (`MultiScaleSpatial.refine`)
-- the direct analog of the old NeuroRaceNet V2 target (`model.amff.refine[0]`,
also the last depthwise-refinement conv in its multi-scale fusion module),
now under the SMGF-Net + MS-CA-ECA architecture.

In [ ]:
class GradCAMPlusPlus:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self.activations = out

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def remove(self):
        self._fwd.remove(); self._bwd.remove()

    def generate(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        out = self.model(input_tensor)
        logits = out["logits"]
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        score = logits[:, class_idx].sum()
        score.backward()

        acts, grads = self.activations, self.gradients
        eps = 1e-8
        g2, g3 = grads ** 2, grads ** 3
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        alpha_denom = 2 * g2 + sum_acts * g3
        alpha_denom = torch.where(alpha_denom.abs() > eps, alpha_denom,
                                   torch.full_like(alpha_denom, eps))
        alpha = g2 / alpha_denom
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)

        cam = F.relu((weights * acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam, class_idx

def default_target_layer(model: nn.Module) -> nn.Module:
    return model.multiscale.refine[0]

def denorm_for_display(tensor_img: torch.Tensor) -> np.ndarray:
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std  = np.array(IMAGENET_STD).reshape(3, 1, 1)
    arr = tensor_img.cpu().numpy() * std + mean
    arr = np.clip(arr, 0, 1).transpose(1, 2, 0)
    return arr

def pick_probe_samples(samples: List[Dict], n_per_class: int = 1, seed: int = SEED) -> List[Dict]:
    rng = random.Random(seed)
    out = []
    for c in range(NUM_CLASSES):
        cls_items = [s for s in samples if s["label"] == c]
        out.extend(rng.sample(cls_items, min(n_per_class, len(cls_items))))
    return out

probe_samples = pick_probe_samples(GLOBAL_TEST_SAMPLES, n_per_class=1)

def show_gradcam_grid(model: nn.Module, probes: List[Dict], title: str, save_path=None):
    fig, axes = plt.subplots(2, len(probes), figsize=(3.2 * len(probes), 6.4))
    target_layer = default_target_layer(model)
    cam_tool = GradCAMPlusPlus(model, target_layer)
    for i, s in enumerate(probes):
        img_t = preprocess(s["path"]).unsqueeze(0).to(DEVICE)
        cam, pred_idx = cam_tool.generate(img_t, class_idx=s["label"])
        disp = denorm_for_display(img_t.squeeze(0))
        axes[0, i].imshow(disp); axes[0, i].set_title(f"true={s['class_name']}"); axes[0, i].axis("off")
        axes[1, i].imshow(disp); axes[1, i].imshow(cam, cmap="jet", alpha=0.45)
        axes[1, i].set_title(f"pred={CLASS_NAMES[pred_idx]}"); axes[1, i].axis("off")
    cam_tool.remove()
    fig.suptitle(title)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

for s in STRATEGIES:
    show_gradcam_grid(trained_models[s], probe_samples, f"Grad-CAM++ — {s}",
                       save_path=str(OUT_DIR / f"fig_gradcampp_{s}.png"))


## §21 · Explainability — Score-CAM

In [ ]:
class ScoreCAM:
    """Gradient-free CAM: masks the input with each upsampled channel activation
    map from the target layer, re-scores the masked input, and uses the softmax
    score as that channel's weight."""
    def __init__(self, model: nn.Module, target_layer: nn.Module, batch_size: int = 16):
        self.model = model
        self.batch_size = batch_size
        self.activations = None
        self._fwd = target_layer.register_forward_hook(self._save_act)

    def _save_act(self, module, inp, out):
        self.activations = out.detach()

    def remove(self):
        self._fwd.remove()

    @torch.no_grad()
    def generate(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None):
        self.model.eval()
        base_out = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(base_out["logits"].argmax(dim=1).item())

        acts = self.activations  # [1, C, h, w]
        C = acts.shape[1]
        H, W = input_tensor.shape[-2:]
        acts_up = F.interpolate(acts, size=(H, W), mode="bilinear", align_corners=False)[0]  # [C,H,W]
        amin = acts_up.view(C, -1).min(dim=1).values.view(C, 1, 1)
        amax = acts_up.view(C, -1).max(dim=1).values.view(C, 1, 1)
        norm_maps = (acts_up - amin) / (amax - amin + 1e-8)  # [C,H,W] in [0,1]

        scores = torch.zeros(C, device=input_tensor.device)
        for start in range(0, C, self.batch_size):
            end = min(start + self.batch_size, C)
            masks = norm_maps[start:end].unsqueeze(1)                 # [b,1,H,W]
            masked_inputs = input_tensor * masks                      # [b,3,H,W]
            out = self.model(masked_inputs)
            probs = torch.softmax(out["logits"], dim=1)
            scores[start:end] = probs[:, class_idx]

        weights = scores.view(C, 1, 1)
        cam = F.relu((weights * acts_up).sum(dim=0))
        cam = cam.detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

def show_scorecam_grid(model: nn.Module, probes: List[Dict], title: str, save_path=None):
    fig, axes = plt.subplots(2, len(probes), figsize=(3.2 * len(probes), 6.4))
    target_layer = default_target_layer(model)
    cam_tool = ScoreCAM(model, target_layer)
    for i, s in enumerate(probes):
        img_t = preprocess(s["path"]).unsqueeze(0).to(DEVICE)
        cam, pred_idx = cam_tool.generate(img_t, class_idx=s["label"])
        disp = denorm_for_display(img_t.squeeze(0))
        axes[0, i].imshow(disp); axes[0, i].set_title(f"true={s['class_name']}"); axes[0, i].axis("off")
        axes[1, i].imshow(disp); axes[1, i].imshow(cam, cmap="jet", alpha=0.45)
        axes[1, i].set_title(f"pred={CLASS_NAMES[pred_idx]}"); axes[1, i].axis("off")
    cam_tool.remove()
    fig.suptitle(title)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

for s in STRATEGIES:
    show_scorecam_grid(trained_models[s], probe_samples, f"Score-CAM — {s}",
                        save_path=str(OUT_DIR / f"fig_scorecam_{s}.png"))


## §22 · Explainability — LIME

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

def make_lime_classifier_fn(model: nn.Module):
    """LIME feeds perturbed RGB uint8 images (H,W,3) already resized to
    IMG_SIZE, from the *AMEF-preprocessed* base image LIME segments. We just
    re-apply the ImageNet normalization used at training time and forward
    through the model, returning per-class softmax probs."""
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1).to(DEVICE)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1).to(DEVICE)
    model.eval()

    @torch.no_grad()
    def classifier_fn(images: np.ndarray) -> np.ndarray:
        batch = torch.tensor(images / 255.0, dtype=torch.float32).permute(0, 3, 1, 2).to(DEVICE)
        batch = (batch - mean) / std
        out = model(batch)
        return torch.softmax(out["logits"], dim=1).cpu().numpy()

    return classifier_fn

def lime_base_image(path: str) -> np.ndarray:
    """AMEF-preprocessed grayscale->RGB uint8 image (before ImageNet norm),
    i.e. what LIME will segment and perturb."""
    gray = read_gray(path)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img01 = robust_normalize(gray)
    if USE_AMEF:
        img01 = amef_preprocess(img01)
    rgb = np.stack([img01, img01, img01], axis=2)
    return (rgb * 255).astype(np.uint8)

def show_lime_grid(model: nn.Module, probes: List[Dict], title: str, save_path=None,
                    num_samples: int = 300):
    explainer = lime_image.LimeImageExplainer()
    classifier_fn = make_lime_classifier_fn(model)
    fig, axes = plt.subplots(2, len(probes), figsize=(3.2 * len(probes), 6.4))
    for i, s in enumerate(probes):
        base_img = lime_base_image(s["path"])
        explanation = explainer.explain_instance(base_img, classifier_fn, top_labels=1,
                                                   hide_color=0, num_samples=num_samples)
        pred_idx = explanation.top_labels[0]
        temp, mask = explanation.get_image_and_mask(pred_idx, positive_only=True,
                                                      num_features=8, hide_rest=False)
        axes[0, i].imshow(base_img); axes[0, i].set_title(f"true={s['class_name']}"); axes[0, i].axis("off")
        axes[1, i].imshow(mark_boundaries(temp / 255.0, mask))
        axes[1, i].set_title(f"pred={CLASS_NAMES[pred_idx]}"); axes[1, i].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

for s in STRATEGIES:
    show_lime_grid(trained_models[s], probe_samples, f"LIME — {s}",
                    save_path=str(OUT_DIR / f"fig_lime_{s}.png"))


## §23 · Summary

Two-way federated aggregation comparison — **FedAvg vs. FedProx
** — on non-IID (Dirichlet-α label-skew) client splits, using the
unchanged SMGF-Net + MS-CA-ECA
classifier (NeuroRaceNet V2 / AMFF / RLDA / ProjectionHead removed). No custom aggregation method, no significance testing, no
ablations.

**Outputs produced:**
- `fedavg_fedprox_fedbn_comparison.csv` — final-round metrics per strategy
- Convergence curves (accuracy/AUC vs. communication round), communication
  cost, strategy comparison bars
- Per-strategy test-set diagnostics: classification report, confusion
  matrix, ROC curves (per-class + micro-avg), Precision-Recall curves
  (per-class + macro-AP), t-SNE of the 64-d embedding space
- Explainability: Grad-CAM++, Score-CAM, and LIME overlays on one probe
  image per class, for each strategy's final global model

## §23.9 · Baseline Replay — run DP without retraining the FL baseline

The DP arm builds its own global model from scratch and needs **nothing** from
the non-private run except the final metrics it is compared against. So the
expensive §15 cell (3 strategies x 50 rounds) can be skipped entirely.

**How to run DP-only:**

1. Run §1-§14 top to bottom. Nothing there trains — they are imports, config,
   the dataset scan/dedup, the model and loss definitions, and the FL helper
   functions.
2. **Skip the §15 training-loop cell** and everything from §16 to §23.
3. Run this cell. It rebuilds the client partition (deterministic from `SEED`,
   so it is the same partition the recorded run used) and loads the recorded
   final metrics from `fedavg_fedprox_fedbn_comparison.csv`, which §15 wrote
   during the original run.
4. Run §24 onward.

**What this cannot restore.** The original §15 saved no model checkpoints —
only that one CSV. So `trained_models` and `test_predictions` are gone, and the
before/after confusion-matrix panel has no baseline predictions to difference
against. The DP confusion matrix is still produced; the baseline one already
exists as a figure from the recorded run. If you want the paired panel, rerun
`run_federated_experiment("fedavg", ...)` alone — one strategy instead of
three, roughly a third of the original baseline cost — which also restores the
per-round history for the convergence plot.

If §15 *has* already run in this session, this cell detects that and does
nothing.


In [ ]:
# ---- reuse the recorded FL baseline instead of retraining it -------------
REPLAY_BASELINE = "fed_results" not in globals()

if not REPLAY_BASELINE:
    print("§15 already ran in this session -- using the live fed_results, nothing replayed.")
else:
    assert len(BRISC_TRAIN_POOL) > 0, "run the §14 data cell first"
    STRATEGIES = ["fedavg", "fedprox"]
    clients_noniid = split_scenario_b_noniid(BRISC_TRAIN_POOL, alpha=NONIID_ALPHA, seed=SEED)
    print(f"Non-IID partition rebuilt from SEED={SEED}: "
          f"{[len(c) for c in clients_noniid]}")

    # Find the CSV that §15 wrote during the recorded run. Order of preference:
    # this session's working dir, then any attached Kaggle dataset / saved
    # notebook output.
    _cands = [OUT_DIR / "fedavg_fedprox_fedbn_comparison.csv"]
    try:
        _cands += sorted(Path("/kaggle/input").glob("**/fedavg_fedprox_fedbn_comparison.csv"))
    except Exception:
        pass
    _csv = next((p for p in _cands if Path(p).exists()), None)
    assert _csv is not None, (
        "fedavg_fedprox_fedbn_comparison.csv not found. It is the only artifact the "
        "recorded §15 run persisted. Attach the saved notebook output (or upload the "
        "CSV) as a Kaggle dataset, or paste the printed results_df table into a "
        "DataFrame here by hand.")
    results_df = pd.read_csv(_csv)
    # The recorded run predates dropping FedBN; its row is discarded here so it
    # never reaches any table or plot. The filename is left unchanged because it
    # is what the recorded run actually wrote.
    _dropped = int((results_df["strategy"] == "fedbn").sum())
    results_df = results_df[results_df["strategy"].isin(STRATEGIES)].reset_index(drop=True)
    print(f"Loaded recorded baseline metrics from: {_csv}"
          + (f"  (discarded {_dropped} fedbn row)" if _dropped else ""))

    # results_df column names -> local_evaluate() metric keys
    _COLMAP = {"acc": "acc", "balanced_acc": "balanced_acc", "precision": "precision_mac",
               "recall": "recall_mac", "f1": "f1_mac", "auc": "auc", "mcc": "mcc",
               "ece": "ece", "mce": "mce", "brier": "brier"}
    fed_results = {}
    for _, _r in results_df.iterrows():
        _fm = {_COLMAP[c]: float(_r[c]) for c in _COLMAP if c in results_df.columns}
        fed_results[_r["strategy"]] = {"strategy": _r["strategy"], "final_metrics": _fm,
                                       "history": {}, "model": None, "replayed": True}
    assert "fedavg" in fed_results, "the recorded CSV has no fedavg row"

    histories = {s: fed_results[s]["history"] for s in fed_results}   # empty: no per-round data
    trained_models = {}                                              # no checkpoints were saved

    print("\nRecorded baseline (not recomputed):")
    print(results_df.to_string(index=False))
    print("\nNot available without a rerun: per-round histories, trained_models, "
          "test_predictions.")


# §24 · Differential Privacy — DP-SGD on the Non-IID Federated Pipeline

Everything above this line is the **non-private baseline** and is left
untouched: same BRISC2025 pool, same dedup, same Dirichlet(alpha=0.3) label-skew
partition object `clients_noniid` built in §15, same held-out
`GLOBAL_TEST_SAMPLES`, same sample-count-weighted aggregation, same 50 rounds
x 2 local epochs, same AdamW(1.2e-4, wd 2e-4), same `SEED`. This section
*adds* a private arm alongside it.

**The partition is reused, not regenerated.** §24.2 takes the exact
`clients_noniid` object produced in §15, prints each client's size and class
composition, and asserts that its per-client path fingerprints match a
re-derived `split_scenario_b_noniid(..., alpha=NONIID_ALPHA, seed=SEED)`. Every
client keeps its own skewed local dataset; nothing is pooled.

**Where DP acts.** Inside local client training only, independently per client:

```
global model -> client k copy -> client k's OWN non-IID local data
                               -> DP-SGD (per-sample grad -> clip to C
                                          -> add N(0, sigma_k^2 C^2 I))
                               -> private update -> weighted average -> new global
```

No noise is added to the aggregated model, to predictions, or to any metric.
This is Stage-2 formal privacy, entirely separate from the Stage-1 Gaussian
input-perturbation robustness experiment; the two are never combined.

**Which strategy the private arm uses: FedAvg.** The baseline runs FedAvg and
FedProx, and the *server-side aggregation is identical in both* —
`federated_aggregate` is sample-count-weighted parameter averaging regardless
of `strategy`. That aggregation is preserved byte-for-byte here (the DP
orchestrator calls the same function). What differs between the two strategies
is local training, and only FedAvg's local step has a valid DP-SGD counterpart:
FedProx adds `(mu/2)||w - w_global||^2` to the local loss, and that term is a
function of the parameter vector rather than of any individual sample, so it
has no per-sample decomposition for Opacus's hooks to attribute and it breaks
the sensitivity bound the accountant assumes.

FedAvg is also the stronger non-IID baseline in §15 (see the §15 table for the current numbers), so the
private arm is measured against the better of the two.

**Four compatibility changes are forced by DP-SGD**, each unavoidable and each
a genuine deviation that must be reported:

1. **BatchNorm -> GroupNorm.** The backbone has 30+ BatchNorm layers. BN
   normalizes using statistics computed *across* the minibatch, so one
   sample's gradient depends on its batch-mates: per-sample gradients are not
   well defined, and BN running statistics are an un-noised function of
   private data that would be transmitted every round. Opacus refuses to train
   such a model. `ModuleValidator.fix()` swaps every BN for
   `GroupNorm(gcd(32, C), C)` — same parameter count, no cross-sample
   coupling, no running buffers. This also happens to matter more here than in
   the IID notebook: the smallest client has 57 training samples, where BN
   batch statistics would be badly estimated anyway. Because this changes the
   normalization, `RUN_GN_CONTROL` in §24.3 runs a non-private GroupNorm
   control that separates the normalization effect from the privacy cost.
2. **`multiscale.scale_weight` is frozen.** It is a bare `nn.Parameter` (3
   scalars) on `MultiScaleSpatial`, a module Opacus has no per-sample grad
   sampler for, so `GradSampleModule` rejects the whole model. Frozen at
   `softmax([1,1,1])`, i.e. equal weighting of F1/F2/F3 — 3 of 211,480
   parameters.
3. **No AMP, no manual `clip_grad_norm_`.** fp16 per-sample gradients underflow
   relative to the clipping norm, and `GradScaler`'s dynamic rescaling breaks
   the fixed sensitivity bound. A second global clip on top of Opacus's
   per-sample clip would silently rescale the calibrated noise.
4. **Loss: two batch-coupling terms removed** (CE normalization and the
   prototype-margin term) — see §24.5.

## §24.1 · Dependency — Opacus

In [ ]:
# Opacus is the only new dependency. --no-deps is deliberate and follows the
# same reasoning as the install cell in §1: opacus's install_requires lets pip
# re-resolve torch/numpy/scipy, and re-resolving those on Kaggle is what
# corrupts the session ("No module named 'numpy.char'"). Everything opacus
# actually needs (torch, numpy, scipy, opt_einsum) already ships in the image.
import subprocess, sys

def _pip_no_deps(pkg: str):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg],
                   check=False)

_pip_no_deps("opacus")
try:
    import opacus
except ModuleNotFoundError as e:
    print(f"transitive dependency missing ({e.name}); installing it without deps")
    _pip_no_deps(e.name.replace("_", "-"))
    import opacus

from opacus import PrivacyEngine, GradSampleModule
from opacus.optimizers import DPOptimizer
from opacus.validators import ModuleValidator
from opacus.accountants import RDPAccountant
from opacus.accountants.utils import get_noise_multiplier

try:
    from opacus.utils.module_utils import trainable_modules
except ImportError:                      # API drift guard
    def trainable_modules(module):
        for m_name, m in module.named_modules():
            if any(p.requires_grad for p in m.parameters(recurse=False)):
                yield m_name, m

print("opacus", opacus.__version__, "| torch", torch.__version__, "| device", DEVICE)

## §24.2 · Verify the Non-IID Partition Is Reused Unchanged

The DP arm must train on the *same* skewed client datasets as the baseline, not
on a fresh Dirichlet draw. This cell takes the `clients_noniid` object built in
§15, fingerprints each client's file list, and re-derives the split from
`split_scenario_b_noniid` to confirm they are identical. It also prints the
per-client class composition matrix, which until now only existed inside
`fig_client_distribution.png`.

In [ ]:
assert "clients_noniid" in globals(), \
    "run §15 first -- the DP arm reuses the exact partition object it builds"

def _partition_fingerprint(clients):
    """md5 over each client's sorted file paths -- identifies a partition
    exactly, without touching the images."""
    return [hashlib.md5("|".join(sorted(s["path"] for s in c)).encode()).hexdigest()[:12]
            for c in clients]

_fp_baseline = _partition_fingerprint(clients_noniid)
_fp_rederived = _partition_fingerprint(
    split_scenario_b_noniid(BRISC_TRAIN_POOL, alpha=NONIID_ALPHA, seed=SEED))
assert _fp_baseline == _fp_rederived, \
    "partition fingerprint mismatch -- the DP arm would train on a different split"

NONIID_CLIENT_SIZES = [len(c) for c in clients_noniid]
_counts = np.zeros((len(clients_noniid), NUM_CLASSES), dtype=int)
for k, c in enumerate(clients_noniid):
    for s in c:
        _counts[k, s["label"]] += 1

noniid_composition_df = pd.DataFrame(_counts, columns=CLASS_NAMES,
                                     index=[f"client {k}" for k in range(len(clients_noniid))])
noniid_composition_df["total"] = NONIID_CLIENT_SIZES
noniid_composition_df["share_of_pool"] = (
    np.array(NONIID_CLIENT_SIZES) / sum(NONIID_CLIENT_SIZES)).round(4)
noniid_composition_df.to_csv(OUT_DIR / "noniid_client_composition.csv")

print(f"Partition fingerprint verified identical to §15: {_fp_baseline}")
print(f"Dirichlet alpha = {NONIID_ALPHA}   clients = {len(clients_noniid)}   "
      f"pool = {sum(NONIID_CLIENT_SIZES)}")
print(noniid_composition_df.to_string())

## §24.2a · Partition report (client sizes, JSD, EMD)

Run before the DP experiment so the sizes that enter the privacy accounting are on record. Reports pairwise JSD (bits) and EMD between clients' label distributions.


In [ ]:
assert "clients_noniid" in globals(), "clients_noniid is missing -- run §15 or the §23.9 replay cell first"
noniid_partition_report = partition_heterogeneity(clients_noniid, "noniid_dirichlet_a0.3")


## §24.3 · DP Configuration

`DP_TARGET_EPSILON` is a *target* used to solve for the noise multiplier via
Opacus's accountant — it is not a claimed result. The epsilon reported at the
end comes from the live `RDPAccountant` after training and is the only number
to quote.

**The non-IID-specific decision: how to calibrate noise across unequal
clients.** Each client runs DP-SGD on its own local dataset, so each has its
own sampling rate `q_k = batch_size / n_k` and its own step count. Those differ
by more than an order of magnitude here. Two defensible calibrations:

* `"uniform_epsilon"` (default) — solve for `sigma_k` per client so **every
  client's data receives the same guarantee**. Small clients get more noise
  (weak subsampling amplification, high `q_k`), large clients less. This is the
  minimum total noise for a given per-client guarantee.
* `"uniform_sigma"` — one `sigma` for all clients, set to the largest value any
  client needs. Every client is then at or below the target, but the large
  clients are heavily over-noised for no privacy benefit.

Either way the headline epsilon reported is the **maximum over clients**, which
is the guarantee that actually holds for every record in the federation.

In [ ]:
# ---- DP hyperparameters -------------------------------------------------
DP_STRATEGY          = "fedavg"      # see §24; server aggregation is unchanged
DP_MAX_GRAD_NORM     = 1.0           # per-sample L2 clipping norm C
DP_DELTA             = 1e-5          # must be < 1/n_train of the SMALLEST client
DP_TARGET_EPSILON    = 8.0           # target used to SOLVE for sigma; not a result
DP_ACCOUNTANT        = "rdp"
DP_CALIBRATION       = "uniform_epsilon"   # or "uniform_sigma" (see §24.3)
DP_ROUNDS            = 50            # independent of COMM_ROUNDS, so extending the baseline does not change the DP run
DP_LOCAL_EPOCHS      = LOCAL_EPOCHS  # unchanged: 2
DP_BATCH_SIZE        = BATCH_SIZE    # unchanged: 32
DP_NUM_WORKERS       = 2             # PrivacyEngine rebuilds the loader every round
DP_POISSON_SAMPLING  = True          # required by the sampled-Gaussian accountant
DP_UNIFORM_CLASS_WEIGHTS = False     # see the class-weight caveat in §26

# Optional, compact extras -- both OFF by default to respect the GPU budget.
DP_RUN_SECOND_CONFIG     = False
DP_SECOND_TARGET_EPSILON = 3.0
RUN_GN_CONTROL           = True      # non-private GroupNorm control (see §24 note 1) -- the control the paper still lacks
RUN_CLIP_ONLY_CONTROL    = False     # per-sample clipping with sigma = 0: separates clipping from noise

print(f"DP config: C={DP_MAX_GRAD_NORM}  delta={DP_DELTA}  target_eps={DP_TARGET_EPSILON}  "
      f"calibration={DP_CALIBRATION}\n           rounds={DP_ROUNDS}  local_epochs={DP_LOCAL_EPOCHS}  "
      f"batch={DP_BATCH_SIZE}  accountant={DP_ACCOUNTANT}")

# delta must be smaller than 1/n for every client's local training set
_min_client_train = min(n - math.ceil(0.15 * n) for n in NONIID_CLIENT_SIZES)
print(f"\nSmallest client local train set: {_min_client_train} samples  "
      f"(1/n = {1/_min_client_train:.5f})")
assert DP_DELTA < 1.0 / _min_client_train, (
    f"DP_DELTA={DP_DELTA} is not < 1/{_min_client_train}; a delta above 1/n makes the "
    f"guarantee vacuous for the smallest client")

# ---- Audit the unmodified model against Opacus --------------------------
_probe = build_smgf_mscaeca()
_bn = [n for n, m in _probe.named_modules()
       if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))]
print(f"\nBatchNorm layers in the baseline model: {len(_bn)}")
_errs = ModuleValidator.validate(_probe, strict=False)
print(f"ModuleValidator errors before fix: {len(_errs)}")
if _errs:
    print("  example:", type(_errs[0]).__name__, str(_errs[0])[:110])
_unsupported = [(n, type(m).__name__) for n, m in trainable_modules(_probe)
                if not GradSampleModule.is_supported(m)]
print("Modules with trainable params and no per-sample grad sampler:", _unsupported)
del _probe

## §24.4 · DP-Compatible Model Builder

`build_dp_model()` reuses `build_smgf_mscaeca()` verbatim — same config dict,
same channel widths, same parameter budget — and then applies only the two
forced fixes from §24. The architecture is not redesigned.

In [ ]:
def build_dp_model(verbose: bool = True, **kwargs) -> nn.Module:
    """SMGF-Net + MS-CA-ECA, made Opacus-compatible.

    Three changes, all required (see §24):
      * ModuleValidator.fix() replaces every BatchNorm{1,2}d with
        GroupNorm(gcd(32, C), C) (Opacus 1.6.0: math.gcd(32, num_features)) --
        identical parameter count (2C), no cross-sample coupling, no running
        buffers to transmit.
      * multiscale.scale_weight (3 scalars) is frozen, because a bare
        nn.Parameter on a module with no registered grad sampler makes
        GradSampleModule reject the whole model. Frozen at softmax([1,1,1]),
        i.e. equal F1/F2/F3 weighting.
      * every in-place activation is made out-of-place, because Opacus's
        backward hooks return views that cannot be modified in place.
    """
    model = build_smgf_mscaeca(**kwargs)
    model = ModuleValidator.fix(model)
    model.multiscale.scale_weight.requires_grad_(False)

    # Opacus attaches full backward hooks, which wrap each module's output in a
    # BackwardHookFunction view. An in-place activation on that view
    # (nn.SiLU(inplace=True) / nn.ReLU(inplace=True)) raises "Output 0 of
    # BackwardHookFunctionBackward is a view and is being modified inplace".
    # Switching them to out-of-place changes memory use only, not the math.
    n_inplace = 0
    for m in model.modules():
        if getattr(m, "inplace", False):
            m.inplace = False
            n_inplace += 1

    model = model.to(DEVICE)

    if verbose:
        n_gn = sum(1 for m in model.modules() if isinstance(m, nn.GroupNorm))
        n_bn = sum(1 for m in model.modules()
                   if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)))
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
        errs = ModuleValidator.validate(model, strict=False)
        unsupported = [(n, type(m).__name__) for n, m in trainable_modules(model)
                       if not GradSampleModule.is_supported(m)]
        print(f"DP model: GroupNorm={n_gn}  BatchNorm={n_bn}  "
              f"trainable={trainable:,}  frozen={frozen:,}  "
              f"in-place activations disabled={n_inplace}")
        from collections import Counter
        _gn = Counter((m.num_channels, m.num_groups) for m in model.modules() if isinstance(m, nn.GroupNorm))
        print("  GroupNorm (channels, groups): count ->",
              {k: v for k, v in sorted(_gn.items())})
        print(f"  ModuleValidator errors: {len(errs)}   unsupported trainable modules: {unsupported}")
        assert not errs and not unsupported, "model is not Opacus-compatible -- stop here"
    return model


_m = build_dp_model()
del _m

## §24.5 · DP-Compatible Loss

DP-SGD's sensitivity argument requires the minibatch loss to be exactly the
mean of independent per-sample losses. Two terms in `CAWLPlus` break that, so
`CAWLPlusDP` overrides only those:

1. **Weighted CE normalization.** `F.cross_entropy(..., weight=w,
   reduction="mean")` divides by `sum(w[y_i])` over the batch — a
   batch-dependent denominator, so sample *i*'s loss depends on its
   batch-mates. Replaced by an explicit per-sample weighted mean (divide by
   *B*). Because `compute_class_weights` normalizes the weights to mean 1, the
   two normalizations agree to within sampling noise. This matters more under
   label skew than it did in the IID notebook: with Dirichlet allocation a
   client's batch can be dominated by one class, so `sum(w[y_i])` swings much
   further from `B`.
2. **Prototype-margin term disabled (`beta = 0`).** The prototypes are updated
   from the raw batch and then read back inside the same loss, which (a)
   couples samples within a batch and (b) is an un-noised statistic of private
   data influencing the released update. This term was already close to inert
   in the baseline: `make_criterion` is called fresh inside the per-round
   client loop, so the prototype buffers reset every round.

The confidence-cap penalty is a clean per-sample mean and is kept unchanged, as
are `label_smoothing`, `target_confidence_cap`, `lambda_max` and the
`anneal_epochs` schedule.

In [ ]:
class CAWLPlusDP(CAWLPlus):
    """CAWL+ with the two batch-coupling terms removed (see §24.5).

    Same class weights, same label smoothing, same confidence-cap penalty and
    same lambda annealing schedule as CAWLPlus; only the CE normalization and
    the prototype-margin term differ.
    """
    def forward(self, logits: torch.Tensor, targets: torch.Tensor,
                emb: Optional[torch.Tensor] = None) -> torch.Tensor:
        w = self.class_weights.to(logits.device)
        ce_i = F.cross_entropy(logits, targets, reduction="none",
                               label_smoothing=self.label_smoothing)
        ce = (w[targets] * ce_i).mean()                      # exact per-sample mean

        probs = F.softmax(logits, dim=1)
        conf, _ = probs.max(dim=1)
        correct = (logits.argmax(dim=1) == targets).float()
        conf_penalty = (F.relu(conf - self.target_confidence_cap) * correct).mean()
        return ce + self._lambda_t() * conf_penalty          # emb intentionally unused


def make_dp_criterion(class_weights: torch.Tensor) -> "CAWLPlusDP":
    """DP analogue of make_criterion(). beta=0 disables the prototype term."""
    return CAWLPlusDP(class_weights,
                      emb_dim=SMGF_MSCAECA_CONFIG['head_hidden'],
                      beta=0.0).to(DEVICE)

print("CAWLPlusDP ready (weighted-CE normalized by batch size, prototype margin off).")

## §24.6 · DP Local Training Step and Per-Client Loaders

`dp_local_train_epoch` is the private counterpart of `local_train_epoch`. The
clipping and the Gaussian noise are **not** written here — they happen inside
`DPOptimizer.step()`, which is what makes them accountable.

Each client keeps its own loader over its own skewed local dataset; nothing is
pooled. The only loader change is `persistent_workers=False` — `PrivacyEngine`
rebuilds the loader with a Poisson batch sampler every round, and 250
persistent worker pools over a full run exhausts Kaggle's shared memory. The
train/val split itself is the baseline's, reused object-for-object.

In [ ]:
def dp_local_train_epoch(dp_model, dp_loader, dp_optimizer, criterion) -> Tuple[float, float]:
    """One local epoch of DP-SGD on ONE client's own non-IID data.

    Differences from local_train_epoch, all required by DP-SGD:
      * no torch.nn.utils.clip_grad_norm_ -- DPOptimizer already clips each
        PER-SAMPLE gradient to max_grad_norm; a second global clip would
        rescale the already-calibrated noise.
      * no AMP/GradScaler -- fp16 per-sample gradients underflow relative to
        the clipping norm and dynamic loss scaling invalidates the fixed
        sensitivity bound the accountant assumes.
      * no FedProx proximal term -- not per-sample decomposable (see §24).
    Clipping + N(0, sigma^2 C^2 I) + the accountant step hook all fire inside
    dp_optimizer.step().
    """
    dp_model.train()
    total_loss = correct = total = 0
    for batch in dp_loader:
        # Poisson sampling can draw an empty batch. For the smallest client
        # (q ~ 0.56, n ~ 57) that probability is ~1e-13, so this is defensive.
        if not isinstance(batch, dict) or batch["image"].shape[0] == 0:
            continue
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)

        dp_optimizer.zero_grad(set_to_none=True)
        out  = dp_model(img)
        loss = criterion(out["logits"], label, out["emb"])
        loss.backward()
        dp_optimizer.step()          # <- clip, noise, and accountant step

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


def make_dp_client_loaders(client_samples: List[Dict], val_frac: float = 0.15,
                           seed: int = SEED) -> Tuple[DataLoader, DataLoader]:
    """Byte-identical train/val split to the baseline -- it literally reuses the
    Datasets that make_client_loaders() builds (same train_test_split call, same
    seed, same stratify-or-fallback logic, which matters here because the
    smallest Dirichlet clients can hold a class with only one sample). Only the
    DataLoader wrapper differs: persistent_workers is disabled because
    PrivacyEngine rebuilds the loader every round.
    """
    base_tr, base_va = make_client_loaders(client_samples, val_frac=val_frac, seed=seed)
    kw = dict(batch_size=DP_BATCH_SIZE, num_workers=DP_NUM_WORKERS,
              pin_memory=(DEVICE == "cuda"), persistent_workers=False)
    return (DataLoader(base_tr.dataset, shuffle=True,  **kw),
            DataLoader(base_va.dataset, shuffle=False, **kw))

print("DP local-training step and per-client loaders ready.")

## §24.7 · Per-Client Noise Calibration

Solved once, up front, against each client's **own** sampling rate and **own**
total step count. Solving per round would calibrate noise for 2 epochs of
exposure while the client actually performs `DP_ROUNDS x DP_LOCAL_EPOCHS`.

This cell is also the diagnostic for the non-IID case: it prints `q_k`, steps,
and `sigma_k` per client and warns where the sampling rate is high enough that
privacy amplification by subsampling barely helps.

In [ ]:
def solve_client_noise(client_data, target_epsilon: float, delta: float,
                       comm_rounds: int, local_epochs: int,
                       calibration: str = None, verbose: bool = True,
                       noise_multiplier_override: Optional[float] = None):
    """Returns (sigmas, sample_rates, total_steps, steps_per_epoch) per client.

    The sampling rate and steps per epoch are read from the same DPDataLoader that
    make_private() builds (DPDataLoader.from_data_loader), not re-derived:
        q = 1 / len(loader) = 1 / ceil(n_train / BATCH_SIZE),   steps/epoch = len(loader).
    The earlier version used q = BATCH_SIZE / n_train and floor(n_train / BATCH_SIZE) steps.
    For the 57-image client that is q = 0.561 and 1 step per epoch against q = 0.5 and 2 steps
    in the real loader, so its noise was calibrated for half the run it executed (achieved
    epsilon 10.51 against a target of 8). The other four clients land within 1 % of target.

    'uniform_epsilon': every client's data gets the same guarantee -> sigma_k differs.
    'uniform_sigma'  : one sigma for all, set to the largest any client needs.
    noise_multiplier_override: fixed sigma for every client (0.0 = clipping-only control).
    """
    from opacus.data_loader import DPDataLoader
    calibration = DP_CALIBRATION if calibration is None else calibration
    assert calibration in ("uniform_epsilon", "uniform_sigma"), calibration

    sigmas, sample_rates, total_steps, loader_steps = [], [], [], []
    for k, (tr_loader, _) in enumerate(client_data):
        probe = DPDataLoader.from_data_loader(tr_loader)
        q, spe = float(probe.sample_rate), len(probe)
        del probe
        steps = comm_rounds * local_epochs * spe
        if noise_multiplier_override is None:
            try:
                sigma = get_noise_multiplier(target_epsilon=target_epsilon,
                                             target_delta=delta, sample_rate=q,
                                             steps=steps, accountant=DP_ACCOUNTANT)
            except TypeError:                              # older opacus: epochs only
                sigma = get_noise_multiplier(target_epsilon=target_epsilon,
                                             target_delta=delta, sample_rate=q,
                                             epochs=comm_rounds * local_epochs,
                                             accountant=DP_ACCOUNTANT)
        else:
            sigma = float(noise_multiplier_override)
        sigmas.append(sigma); sample_rates.append(q)
        total_steps.append(steps); loader_steps.append(spe)

    if calibration == "uniform_sigma" and noise_multiplier_override is None:
        sigmas = [max(sigmas)] * len(sigmas)

    if verbose:
        print(f"  calibration = {calibration}, target_eps = {target_epsilon}, delta = {delta}")
        print(f"  {'k':>2}{'n_train':>9}{'q':>9}{'steps/ep':>10}{'steps':>8}{'sigma':>9}")
        for k in range(len(sigmas)):
            n_train = len(client_data[k][0].dataset)
            print(f"  {k:>2}{n_train:>9}{sample_rates[k]:>9.4f}{loader_steps[k]:>10}"
                  f"{total_steps[k]:>8}{sigmas[k]:>9.4f}")
            if sample_rates[k] > 0.2:
                print(f"      [warn] client {k}: q={sample_rates[k]:.3f} -- batch is a large "
                      f"fraction of this client's data, so privacy amplification by "
                      f"subsampling is weak and sigma must be correspondingly large.")
    return sigmas, sample_rates, total_steps, loader_steps

print("solve_client_noise() ready.")


## §24.8 · DP Federated Orchestrator

Mirrors `run_federated_experiment` round-for-round. The server side is the
**existing** `federated_aggregate` / `weighted_average_state_dicts`, called with
the same `client_n` sample-count weights — under this partition those weights
are highly unequal (client shares span roughly 1.4% to 43%), and that imbalance
is preserved exactly as in the baseline.

**Accounting across rounds.** Each client keeps *one* `RDPAccountant` for the
whole run, injected into the per-round `PrivacyEngine` before `make_private()`,
so the optimizer's step hook appends every local step to that client's history.
`get_epsilon()` at the end therefore covers all of that client's participation,
not a single round.

In [ ]:
def run_dp_federated_experiment(client_partitions: List[List[Dict]],
                                global_test_samples: List[Dict],
                                target_epsilon: float = None,
                                max_grad_norm: float = None,
                                delta: float = None,
                                comm_rounds: int = None,
                                local_epochs: int = None,
                                calibration: str = None,
                                seed: int = SEED, verbose_every: int = 5,
                                noise_multiplier_override: Optional[float] = None,
                                tag: str = "dp_fedavg") -> Dict[str, Any]:
    """DP-FedAvg on the non-IID partition. Same client datasets, same
    aggregation, same test set as the baseline; local training replaced by
    DP-SGD, run independently on each client's own skewed local data."""
    target_epsilon = DP_TARGET_EPSILON  if target_epsilon is None else target_epsilon
    max_grad_norm  = DP_MAX_GRAD_NORM   if max_grad_norm  is None else max_grad_norm
    delta          = DP_DELTA           if delta          is None else delta
    comm_rounds    = DP_ROUNDS          if comm_rounds    is None else comm_rounds
    local_epochs   = DP_LOCAL_EPOCHS    if local_epochs   is None else local_epochs
    calibration    = DP_CALIBRATION     if calibration    is None else calibration

    # Re-seed so the private run is reproducible independently of how many RNG
    # draws the baseline consumed earlier in the session. Same SEED as §2.
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    n_clients = len(client_partitions)
    global_model = build_dp_model()

    if DP_UNIFORM_CLASS_WEIGHTS:
        class_weights = torch.ones(NUM_CLASSES, dtype=torch.float32)
    else:
        # identical to the baseline: pooled over all clients (see §26 caveat)
        class_weights = compute_class_weights([s for c in client_partitions for s in c])

    client_data = [make_dp_client_loaders(c, seed=seed) for c in client_partitions]
    client_n    = [len(c) for c in client_partitions]          # unequal, as in §15
    test_loader = DataLoader(BrainTumorDataset(global_test_samples, is_train=False),
                             **_loader_kwargs(shuffle=False))

    sigmas, sample_rates, total_steps, loader_steps = solve_client_noise(
        client_data, target_epsilon, delta, comm_rounds, local_epochs, calibration,
        noise_multiplier_override=noise_multiplier_override)
    no_noise = (noise_multiplier_override is not None and float(noise_multiplier_override) == 0.0)

    accountants = {k: RDPAccountant() for k in range(n_clients)}

    history = {"round": [], "acc": [], "auc": [], "ece": [], "mce": [], "brier": [],
               "f1_mac": [], "balanced_acc": [], "epsilon": [], "comm_bytes_round": []}
    client_history: Dict[int, List[Dict]] = {k: [] for k in range(n_clients)}
    param_bytes = sum(p.numel() for p in global_model.parameters()) * 4

    for rnd in range(1, comm_rounds + 1):
        client_states = []
        for k in range(n_clients):
            local_model = copy.deepcopy(global_model)
            # local_evaluate() leaves global_model in eval mode after every
            # round, so from round 2 on the deepcopy arrives in eval mode and
            # PrivacyEngine.make_private() rejects it ("Model needs to be in
            # training mode"). Put the client copy back in train mode first.
            local_model.train()
            criterion = make_dp_criterion(class_weights)
            criterion.current_epoch = rnd
            optimizer = torch.optim.AdamW(
                [p for p in local_model.parameters() if p.requires_grad],
                lr=CLIENT_LR, weight_decay=CLIENT_WEIGHT_DECAY)

            engine = PrivacyEngine(accountant=DP_ACCOUNTANT)
            if not no_noise:
                engine.accountant = accountants[k]      # carry history across rounds
            dp_model, dp_optimizer, dp_loader = engine.make_private(
                module=local_model,
                optimizer=optimizer,
                data_loader=client_data[k][0],          # THIS client's own data only
                noise_multiplier=sigmas[k],
                max_grad_norm=max_grad_norm,
                poisson_sampling=DP_POISSON_SAMPLING,
            )
            # Guard against the classic bug: training with the plain optimizer
            # while a DPOptimizer sits unused.
            assert isinstance(dp_optimizer, DPOptimizer), "DPOptimizer was not attached"
            assert isinstance(dp_model, GradSampleModule), "per-sample grads not enabled"
            if rnd == 1:
                # the calibration inputs must be what the loader really does
                assert len(dp_loader) == loader_steps[k], (
                    f"client {k}: solver assumed {loader_steps[k]} steps/epoch, loader has {len(dp_loader)}")
                assert abs(dp_loader.sample_rate - sample_rates[k]) < 1e-12, (
                    f"client {k}: solver q={sample_rates[k]}, loader q={dp_loader.sample_rate}")
            if rnd == 1:
                print(f"  [check] client {k}: noise_multiplier={dp_optimizer.noise_multiplier:.4f} "
                      f"max_grad_norm={dp_optimizer.max_grad_norm} "
                      f"expected_batch_size={dp_optimizer.expected_batch_size} "
                      f"len(dp_loader)={len(dp_loader)}")

            for _ in range(local_epochs):
                dp_local_train_epoch(dp_model, dp_loader, dp_optimizer, criterion)

            inner = dp_model.to_standard_module()       # unwrap + detach hooks
            client_states.append({kk: vv.clone() for kk, vv in inner.state_dict().items()})
            m_local = local_evaluate(inner, client_data[k][1], criterion)
            client_history[k].append({"round": rnd, "balanced_acc": m_local["balanced_acc"],
                                      "f1_mac": m_local["f1_mac"]})
            del dp_model, dp_optimizer, dp_loader, local_model, inner, engine

        # ---- server side: the EXISTING aggregation, unchanged --------------
        agg_state, alphas = federated_aggregate(DP_STRATEGY, client_states, client_n)
        gsd = global_model.state_dict()
        gsd.update(agg_state)
        global_model.load_state_dict(gsd)

        eval_criterion = make_dp_criterion(class_weights); eval_criterion.current_epoch = rnd
        m = local_evaluate(global_model, test_loader, eval_criterion)
        eps_now = (float("inf") if no_noise else
                   max(accountants[k].get_epsilon(delta=delta) for k in range(n_clients)))

        history["round"].append(rnd)
        history["acc"].append(m["acc"]); history["auc"].append(m["auc"])
        history["ece"].append(m["ece"]); history["mce"].append(m["mce"])
        history["brier"].append(m["brier"]); history["f1_mac"].append(m["f1_mac"])
        history["balanced_acc"].append(m["balanced_acc"])
        history["epsilon"].append(eps_now)
        history["comm_bytes_round"].append(2 * n_clients * param_bytes)

        if rnd % verbose_every == 0 or rnd == 1:
            print(f"[{tag}] round {rnd:03d}  acc={m['acc']:.4f}  auc={m['auc']:.4f}  "
                  f"f1={m['f1_mac']:.4f}  eps={eps_now:.3f}")
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    per_client_eps = {k: (float("inf") if no_noise else accountants[k].get_epsilon(delta=delta))
                      for k in range(n_clients)}
    eps_max = max(per_client_eps.values())
    if not no_noise:
        print(f"  achieved epsilon {eps_max:.4f} vs target {target_epsilon:g}  "
              f"(ratio {eps_max / target_epsilon:.3f}); per client "
              f"{ {k: round(v, 3) for k, v in per_client_eps.items()} }")
        if eps_max > 1.02 * target_epsilon:
            print("  WARNING: achieved epsilon exceeds the target by more than 2 %")
    privacy = {
        "mechanism": "DP-SGD (Opacus PrivacyEngine, sampled-Gaussian)",
        "accountant": DP_ACCOUNTANT,
        "opacus_version": opacus.__version__,
        "calibration": calibration,
        "target_epsilon": target_epsilon,
        "epsilon_per_client": per_client_eps,
        "epsilon": eps_max,
        "delta": delta,
        "noise_multiplier": sigmas,
        "max_grad_norm": max_grad_norm,
        "sample_rate": sample_rates,
        "total_steps_per_client": total_steps,
        "loader_steps_per_epoch": loader_steps,
        "solver_matches_loader": True,          # asserted in round 1; old runs lack this key
        "epsilon_over_target": (None if no_noise else eps_max / target_epsilon),
        "noise_multiplier_override": noise_multiplier_override,
        "class_weights_noised": False,          # omega_c is computed from pooled client labels, un-noised, outside the accountant
        "client_sizes": client_n,
        "aggregation_weights": [float(a) for a in alphas],
        "comm_rounds": comm_rounds,
        "local_epochs": local_epochs,
        "batch_size": DP_BATCH_SIZE,
        "clients": n_clients,
        "partition": f"Dirichlet(alpha={NONIID_ALPHA}) label skew, reused from §15",
        "client_participation": "all clients every round (full participation)",
        "granularity": "sample-level DP per client",
        "poisson_sampling": DP_POISSON_SAMPLING,
        "secure_mode": False,
    }
    return {"strategy": tag, "model": global_model, "history": history,
            "client_history": client_history, "final_metrics": m, "privacy": privacy}

print("run_dp_federated_experiment() ready.")

## §24.9 · Run the DP Experiment

Expect this to be noticeably slower than the baseline arm: per-sample gradients
cost roughly 2-3x a normal backward pass, and AMP is off.

In [ ]:
print(f"=== DP-FedAvg on Dirichlet(alpha={NONIID_ALPHA}) clients  "
      f"(target_eps={DP_TARGET_EPSILON}, delta={DP_DELTA}, C={DP_MAX_GRAD_NORM}) ===")
dp_result = run_dp_federated_experiment(clients_noniid, GLOBAL_TEST_SAMPLES,
                                        tag="dp_fedavg_noniid", seed=SEED)

dp_priv = dp_result["privacy"]
print("\n---- PRIVACY REPORT (from the accountant, after training) ----")
for key in ["mechanism", "accountant", "granularity", "partition", "calibration",
            "delta", "max_grad_norm", "batch_size", "comm_rounds", "local_epochs",
            "clients", "client_participation", "poisson_sampling", "secure_mode"]:
    print(f"  {key:24s}: {dp_priv[key]}")
print(f"  {'client_sizes':24s}: {dp_priv['client_sizes']}")
print(f"  {'aggregation_weights':24s}: {[round(a, 4) for a in dp_priv['aggregation_weights']]}")
print(f"  {'noise_multiplier':24s}: {[round(s, 4) for s in dp_priv['noise_multiplier']]}")
print(f"  {'sample_rate':24s}: {[round(q, 4) for q in dp_priv['sample_rate']]}")
print(f"  {'steps per client':24s}: {dp_priv['total_steps_per_client']}")
print(f"  {'epsilon per client':24s}: "
      f"{ {k: round(v, 4) for k, v in dp_priv['epsilon_per_client'].items()} }")
print(f"  {'EPSILON (max, reported)':24s}: {dp_priv['epsilon']:.4f}  at delta={dp_priv['delta']}")

assert np.isfinite(dp_priv["epsilon"]), \
    "accountant returned a non-finite epsilon -- do not report a privacy claim from this run"
print(f"  {'epsilon / target':24s}: {dp_priv['epsilon'] / dp_priv['target_epsilon']:.3f}")

## §24.10 · Optional Second Privacy Setting and GroupNorm Control

Both off by default. The second setting adds one more run at a tighter target
epsilon. The GroupNorm control runs `run_federated_experiment` with the DP arm's
builder, loss (`CAWLPlusDP`) and AMP off, with no noise and no per-sample
clipping, which separates the cost of removing BatchNorm from the cost of
privacy — without it, the baseline-vs-DP gap conflates the two.

In [ ]:
dp_result_2 = None
if DP_RUN_SECOND_CONFIG:
    print(f"=== DP-FedAvg (target_eps={DP_SECOND_TARGET_EPSILON}) ===")
    dp_result_2 = run_dp_federated_experiment(
        clients_noniid, GLOBAL_TEST_SAMPLES,
        target_epsilon=DP_SECOND_TARGET_EPSILON,
        tag=f"dp_fedavg_noniid_eps{DP_SECOND_TARGET_EPSILON:g}", seed=SEED)
    p2 = dp_result_2["privacy"]
    print(f"\n  epsilon={p2['epsilon']:.4f}  delta={p2['delta']}  "
          f"sigma={[round(s, 4) for s in p2['noise_multiplier']]}")
else:
    print("DP_RUN_SECOND_CONFIG = False -- skipping the second privacy setting.")

gn_control = None
clip_only_control = None

if RUN_GN_CONTROL:
    # Non-private run of the DP arm's architecture: GroupNorm (gcd rule), frozen scale_weight,
    # out-of-place activations, CAWLPlusDP (no prototype term, exact per-sample mean), AMP off.
    # Nothing is clipped per sample and no noise is added; the client update still goes through
    # the baseline's global-norm clip (1.0) in local_train_epoch. Compared with the BatchNorm
    # baseline it isolates normalization + loss + AMP; compared with the DP arm it isolates
    # clipping + noise.
    print("=== non-private GroupNorm control (FedAvg, non-IID) ===")
    gn_control = run_federated_experiment(
        "fedavg", clients_noniid, GLOBAL_TEST_SAMPLES, seed=SEED,
        model_builder=lambda: build_dp_model(verbose=False),
        criterion_factory=make_dp_criterion, use_amp=False,
        tag="fedavg_groupnorm_control")
    gm = gn_control["final_metrics"]
    print(f"\nGroupNorm control: acc={gm['acc']:.4f}  f1={gm['f1_mac']:.4f}  auc={gm['auc']:.4f}")
else:
    print("RUN_GN_CONTROL = False -- skipping the GroupNorm control.")

if RUN_CLIP_ONLY_CONTROL:
    # DP-SGD machinery with sigma = 0: per-sample clipping to C, no noise, no epsilon.
    print("=== per-sample clipping only (sigma = 0) ===")
    clip_only_control = run_dp_federated_experiment(
        clients_noniid, GLOBAL_TEST_SAMPLES, noise_multiplier_override=0.0,
        tag="dp_clip_only_control", seed=SEED)
    cm_ = clip_only_control["final_metrics"]
    print(f"\nClip-only control: acc={cm_['acc']:.4f}  f1={cm_['f1_mac']:.4f}  auc={cm_['auc']:.4f}")
else:
    print("RUN_CLIP_ONLY_CONTROL = False -- skipping the clipping-only control.")


# §25 · Privacy-Utility Comparison (Non-IID)

Baseline numbers are read out of the `fed_results` / `results_df` objects
produced in §15 — nothing there is recomputed or overwritten. Only metrics the
notebook already computes in `local_evaluate` appear in the table.

New artifacts are written with a `noniid_` prefix so they do not collide with
the IID notebook's output filenames in `/kaggle/working`.

In [ ]:
def _row(name, privacy_label, metrics, priv=None):
    return {
        "experiment": name,
        "privacy": privacy_label,
        "epsilon": (priv["epsilon"] if priv else np.inf),
        "delta": (priv["delta"] if priv else np.nan),
        "noise_multiplier_mean": (round(float(np.mean(priv["noise_multiplier"])), 4) if priv else np.nan),
        "clipping_norm": (priv["max_grad_norm"] if priv else np.nan),
        "accuracy": metrics["acc"],
        "balanced_acc": metrics["balanced_acc"],
        "macro_f1": metrics["f1_mac"],
        "auc": metrics["auc"],
        "ece": metrics["ece"],
        "brier": metrics["brier"],
    }

rows_pu = [_row("Non-IID + FL (FedAvg)", "Non-private",
                fed_results["fedavg"]["final_metrics"])]
if "fedprox" in fed_results:
    rows_pu.append(_row("Non-IID + FL (FedProx, reference)", "Non-private",
                        fed_results["fedprox"]["final_metrics"]))
if gn_control is not None:
    rows_pu.append(_row("Non-IID + FL (FedAvg, GroupNorm control)", "Non-private",
                        gn_control["final_metrics"]))
if clip_only_control is not None:
    rows_pu.append(_row("Non-IID + FL (GroupNorm, per-sample clipping, no noise)", "Clip only",
                        clip_only_control["final_metrics"]))
rows_pu.append(_row(f"Non-IID + FL + DP (target eps={DP_TARGET_EPSILON:g})", "DP-SGD",
                    dp_result["final_metrics"], dp_result["privacy"]))
if dp_result_2 is not None:
    rows_pu.append(_row(f"Non-IID + FL + DP (target eps={DP_SECOND_TARGET_EPSILON:g})", "DP-SGD",
                        dp_result_2["final_metrics"], dp_result_2["privacy"]))

privacy_utility_df = pd.DataFrame(rows_pu)
privacy_utility_df.to_csv(OUT_DIR / "noniid_fl_dp_privacy_utility.csv", index=False)

pd.set_option("display.width", 220)
print(privacy_utility_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

base = fed_results["fedavg"]["final_metrics"]
dpm  = dp_result["final_metrics"]
print("\nDelta vs. the non-private non-IID FedAvg baseline "
      "(negative = DP is worse; reported as-is):")
for key, label in [("acc", "accuracy"), ("f1_mac", "macro-F1"), ("auc", "AUC"),
                   ("balanced_acc", "balanced acc")]:
    print(f"  {label:14s} {base[key]:.4f} -> {dpm[key]:.4f}   ({dpm[key] - base[key]:+.4f})")

# per-client privacy table
priv_tbl = pd.DataFrame({
    "client": list(range(len(dp_priv["client_sizes"]))),
    "n_total": dp_priv["client_sizes"],
    "sample_rate_q": [round(q, 4) for q in dp_priv["sample_rate"]],
    "steps_per_epoch": dp_priv["loader_steps_per_epoch"],
    "local_steps": dp_priv["total_steps_per_client"],
    "noise_multiplier": [round(s, 4) for s in dp_priv["noise_multiplier"]],
    "epsilon": [round(dp_priv["epsilon_per_client"][k], 4)
                for k in range(len(dp_priv["client_sizes"]))],
    "agg_weight": [round(a, 4) for a in dp_priv["aggregation_weights"]],
})
priv_tbl.to_csv(OUT_DIR / "noniid_dp_per_client_privacy.csv", index=False)
print("\nPer-client privacy accounting:")
print(priv_tbl.to_string(index=False))

## §25.1 · Privacy-Utility Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

# (a) convergence: baseline vs DP
ax = axes[0]
_base_hist = histories.get("fedavg", {}) if "histories" in globals() else {}
if _base_hist.get("round"):
    ax.plot(_base_hist["round"], _base_hist["acc"],
            label="FedAvg (non-private)", linewidth=2)
else:
    # §15 was not rerun in this session -- only the recorded final metric exists
    ax.axhline(fed_results["fedavg"]["final_metrics"]["acc"], ls="--", color="gray",
               label="FedAvg final (recorded)")
if "histories" in globals() and histories.get("fedprox", {}).get("round"):
    ax.plot(histories["fedprox"]["round"], histories["fedprox"]["acc"],
            ":", label="FedProx (non-private)", linewidth=1.6)
ax.plot(dp_result["history"]["round"], dp_result["history"]["acc"],
        label=f"DP-FedAvg (eps={dp_result['privacy']['epsilon']:.2f})", linewidth=2)
if dp_result_2 is not None:
    ax.plot(dp_result_2["history"]["round"], dp_result_2["history"]["acc"],
            label=f"DP-FedAvg (eps={dp_result_2['privacy']['epsilon']:.2f})", linewidth=2)
if gn_control is not None:
    ax.plot(gn_control["history"]["round"], gn_control["history"]["acc"],
            "--", label="GroupNorm control", linewidth=1.6)
ax.set_xlabel("Communication round"); ax.set_ylabel("Global test accuracy")
ax.set_title(f"Convergence — Dirichlet(alpha={NONIID_ALPHA})")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# (b) privacy budget spent per round
ax = axes[1]
ax.plot(dp_result["history"]["round"], dp_result["history"]["epsilon"], linewidth=2,
        label="max over clients")
if dp_result_2 is not None:
    ax.plot(dp_result_2["history"]["round"], dp_result_2["history"]["epsilon"], linewidth=2)
ax.set_xlabel("Communication round"); ax.set_ylabel(f"epsilon (delta={DP_DELTA:g})")
ax.set_title("Cumulative privacy budget"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# (c) per-client sigma against client size -- the non-IID DP cost
ax = axes[2]
ax.bar([f"c{k}\nn={n}" for k, n in enumerate(dp_priv["client_sizes"])],
       dp_priv["noise_multiplier"],
       color=plt.cm.Set2(np.linspace(0, 1, len(dp_priv["client_sizes"]))))
ax.set_ylabel("noise multiplier sigma")
ax.set_title(f"Noise needed per client for eps<={DP_TARGET_EPSILON:g}")
plt.setp(ax.get_xticklabels(), fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig_noniid_privacy_utility.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n===== DP-FedAvg (non-IID) — Classification Report "
      f"(same test set, n={len(GLOBAL_TEST_SAMPLES)}) =====")
dp_preds = collect_test_predictions(dp_result["model"], GLOBAL_TEST_SAMPLES)
print(classification_report(dp_preds["y_true"], dp_preds["y_pred"],
                            target_names=CLASS_NAMES, digits=4))
plot_confusion_matrix(dp_preds["y_true"], dp_preds["y_pred"],
                      "Confusion Matrix — DP-FedAvg (non-IID)",
                      save_path=str(OUT_DIR / "fig_noniid_confusion_dp_fedavg.png"))

### Confusion matrix — before vs. after DP

Same held-out test set for both panels, so the cells are directly comparable.
The third panel is the signed difference in raw counts: it shows which class
pairs the DP noise actually pushed images between, rather than only that
accuracy moved.


In [ ]:
# ---- Confusion matrix: non-private FedAvg vs DP-FedAvg, side by side -----
_have_base = "test_predictions" in globals() and "fedavg" in globals().get("test_predictions", {})

def _before_after_cm():
    base_preds = test_predictions["fedavg"]
    cm_base = confusion_matrix(base_preds["y_true"], base_preds["y_pred"])
    cm_dp   = confusion_matrix(dp_preds["y_true"],  dp_preds["y_pred"])
    cm_diff = cm_dp - cm_base          # positive = DP puts MORE images in that cell

    acc_base = float((base_preds["y_pred"] == base_preds["y_true"]).mean())
    acc_dp   = float((dp_preds["y_pred"]  == dp_preds["y_true"]).mean())

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

    def _draw(ax, cm, title, cmap, diverging=False):
        vmax = max(int(np.abs(cm).max()), 1)
        im = ax.imshow(cm, cmap=cmap, vmin=(-vmax if diverging else 0), vmax=vmax)
        ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
        ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
        ax.set_yticklabels(CLASS_NAMES)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=10)
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                v = int(cm[i, j])
                txt = f"{v:+d}" if (diverging and v != 0) else str(v)
                ax.text(j, i, txt, ha="center", va="center", fontsize=9,
                        color="white" if abs(v) > vmax / 2 else "black")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    _draw(axes[0], cm_base, f"Before DP — FedAvg (non-IID)\nacc={acc_base:.4f}", "Blues")
    _draw(axes[1], cm_dp,
          f"After DP — DP-FedAvg (eps={dp_priv['epsilon']:.2f})\nacc={acc_dp:.4f}", "Blues")
    _draw(axes[2], cm_diff, "Difference (DP - baseline)", "RdBu_r", diverging=True)

    plt.tight_layout()
    plt.savefig(OUT_DIR / "fig_noniid_confusion_before_after_dp.png", dpi=150, bbox_inches="tight")
    plt.show()
    return cm_base, cm_dp


def _per_class(cm):
    tp = np.diag(cm).astype(float)
    return tp / np.maximum(cm.sum(axis=1), 1), tp / np.maximum(cm.sum(axis=0), 1)


if _have_base:
    cm_base, cm_dp = _before_after_cm()
    r_b, p_b = _per_class(cm_base)
    r_d, p_d = _per_class(cm_dp)
    per_class_df = pd.DataFrame({
        "class": CLASS_NAMES, "support": cm_base.sum(axis=1),
        "recall_before": r_b.round(4), "recall_after": r_d.round(4),
        "recall_delta": (r_d - r_b).round(4),
        "precision_before": p_b.round(4), "precision_after": p_d.round(4),
        "precision_delta": (p_d - p_b).round(4),
    })
    print(f"Per-class effect of DP (same test set, n={len(dp_preds['y_true'])}):")
else:
    # Baseline predictions need the trained non-private FedAvg model, and the
    # recorded §15 run saved no checkpoint. DP-only table instead.
    cm_dp = confusion_matrix(dp_preds["y_true"], dp_preds["y_pred"])
    r_d, p_d = _per_class(cm_dp)
    per_class_df = pd.DataFrame({
        "class": CLASS_NAMES, "support": cm_dp.sum(axis=1),
        "recall_after": r_d.round(4), "precision_after": p_d.round(4),
    })
    print("Baseline test predictions are not in memory, so the before/after panel is\n"
          "skipped. It needs the trained non-private FedAvg model and the recorded run\n"
          "saved no checkpoint; rerun run_federated_experiment('fedavg', ...) alone to\n"
          "regenerate it. DP-only per-class numbers:")

per_class_df.to_csv(OUT_DIR / "noniid_dp_per_class_effect.csv", index=False)
print(per_class_df.to_string(index=False))


## §25.2 · Persist DP Artifacts

In [ ]:
import json as _json

torch.save(dp_result["model"].state_dict(), OUT_DIR / "noniid_dp_fedavg_global_model.pt")
pd.DataFrame(dp_result["history"]).to_csv(OUT_DIR / "noniid_dp_fedavg_history.csv", index=False)

_priv = dict(dp_result["privacy"])
_priv["epsilon_per_client"] = {str(k): v for k, v in _priv["epsilon_per_client"].items()}
with open(OUT_DIR / "noniid_dp_privacy_report.json", "w") as f:
    _json.dump(_priv, f, indent=2)

_persist_rows = [persist_run(dp_result, "noniid_dp_fedavg")]
if dp_result_2 is not None:
    _persist_rows.append(persist_run(dp_result_2, f"noniid_dp_fedavg_eps{DP_SECOND_TARGET_EPSILON:g}"))
if gn_control is not None:
    _persist_rows.append(persist_run(gn_control, "noniid_fl_fedavg_groupnorm_control"))
if clip_only_control is not None:
    _persist_rows.append(persist_run(clip_only_control, "noniid_dp_clip_only_control"))
_persist_rows = [r for r in _persist_rows if r is not None]
if _persist_rows:
    pd.DataFrame(_persist_rows).to_csv(OUT_DIR / "noniid_dp_run_summary.csv", index=False)

print("Saved: noniid_dp_fedavg_global_model.pt, noniid_dp_fedavg_history.csv,")
print("       noniid_dp_privacy_report.json, noniid_dp_per_client_privacy.csv,")
print("       noniid_fl_dp_privacy_utility.csv, noniid_client_composition.csv,")
print("       fig_noniid_privacy_utility.png, fig_noniid_confusion_dp_fedavg.png")
print("\nBaseline artifacts from §15 untouched:",
      (OUT_DIR / "fedavg_fedprox_fedbn_comparison.csv").exists())

## §25.3 · Privacy-Level Ablation — strong / moderate / weak

Three target privacy budgets at the same δ, clipping norm, rounds, local
epochs, partition and test set; only the noise multiplier changes, and it is
solved by the accountant for each target. The labels are conventional — the
numbers are what should be reported:

| level | target ε | δ |
|---|---|---|
| strong | 1.0 | `DP_DELTA` |
| moderate | 3.0 | `DP_DELTA` |
| weak | 20.0 | `DP_DELTA` |

The reported ε for each level is the accountant's value after training, not
the target.

**ε = 8 is the main DP operating point.** It is run in §24.7 and added to the
summary table below as a reference row (it is not one of the three levels), so
the ablation table shows the whole range from 1 to 20 in one place.
 Each level below is saved to disk the
moment it finishes, and any level already on disk (in `/kaggle/working` or
an attached `/kaggle/input` dataset) is loaded instead of retrained, so the
levels can be spread across separate Kaggle sessions: set
`EPS_LEVELS_TO_RUN` to one level per session, attach the previous sessions'
outputs, and the summary cell below assembles whatever exists.

**Running this in a fresh session without retraining anything else:** run
§1–§14, the §23.9 baseline replay, and the definition cells §24.1–§24.6; skip
the §24.7 run cell and §24.8–§25.2; then run the two cells below.


In [ ]:
# ---- epsilon ablation, part 1: obtain every level (load if possible, else train)
import json

EPS_ABLATION = {              # level -> target epsilon (delta = DP_DELTA for every level)
    "strong":   1.0,
    "moderate": 3.0,
    "weak":    20.0,
}
# Levels this session is allowed to TRAIN. To split the work across Kaggle
# sessions, put one level here per session, e.g. ["strong"], then
# ["moderate"], then ["weak"].
EPS_LEVELS_TO_RUN = ["strong", "moderate", "weak"]

MAIN_RUN_HISTORY = "noniid_dp_fedavg_history.csv"   # written by §25.2
MAIN_RUN_PRIVACY = "noniid_dp_privacy_report.json"


def _find_artifact(name):
    """This session's working dir first, then any attached Kaggle input."""
    p = OUT_DIR / name
    if p.exists():
        return p
    try:
        hits = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    except Exception:
        hits = []
    return hits[0] if hits else None


def _load_run(hist_name, priv_name, target):
    """Rebuild a finished DP run from disk. final_metrics is the last history
    row, which is exactly the final-round local_evaluate() result."""
    h, p = _find_artifact(hist_name), _find_artifact(priv_name)
    if h is None or p is None:
        return None
    hist_df = pd.read_csv(h)
    with open(p) as f:
        priv = json.load(f)
    # refuse to mix in a run made under a different configuration
    _checks = {
        "target_epsilon": abs(priv["target_epsilon"] - target) < 1e-9,
        "max_grad_norm":  priv["max_grad_norm"] == DP_MAX_GRAD_NORM,
        "delta":          priv["delta"] == DP_DELTA,
        "comm_rounds":    priv["comm_rounds"] == DP_ROUNDS,
        "local_epochs":   priv["local_epochs"] == DP_LOCAL_EPOCHS,
        "complete":       len(hist_df) == priv["comm_rounds"],
        # runs saved before the solver/loader fix lack this key and must not be mixed in
        "solver_matches_loader": bool(priv.get("solver_matches_loader", False)),
    }
    bad = [k for k, ok in _checks.items() if not ok]
    if bad:
        print(f"   found {h.name} but it does not match this configuration "
              f"({', '.join(bad)}); ignoring it")
        return None
    last = hist_df.iloc[-1]
    fm = {k: float(last[k]) for k in
          ["acc", "auc", "ece", "mce", "brier", "f1_mac", "balanced_acc"]}
    return {"history": {c: hist_df[c].tolist() for c in hist_df.columns},
            "final_metrics": fm, "privacy": priv, "model": None,
            "source": str(h)}


def _save_level(level, res):
    pd.DataFrame(res["history"]).to_csv(
        OUT_DIR / f"noniid_eps_ablation_{level}_history.csv", index=False)
    _priv = dict(res["privacy"])
    _priv["epsilon_per_client"] = {str(k): v for k, v in _priv["epsilon_per_client"].items()}
    with open(OUT_DIR / f"noniid_eps_ablation_{level}_privacy.json", "w") as f:
        json.dump(_priv, f, indent=2)
    if res.get("model") is not None:
        torch.save(res["model"].state_dict(),
                   OUT_DIR / f"noniid_eps_ablation_{level}_model.pt")


if "eps_ablation_results" not in globals():
    eps_ablation_results = {}

for level, target in EPS_ABLATION.items():
    if level in eps_ablation_results:
        print(f"[{level:8s}] eps={target:g}: already in memory")
        continue

    # 1) the §24.7 run, if this is its level and it is still in memory
    _main = globals().get("dp_result")
    if (_main is not None
            and abs(_main["privacy"]["target_epsilon"] - target) < 1e-9
            and _main["privacy"]["max_grad_norm"] == DP_MAX_GRAD_NORM
            and _main["privacy"]["delta"] == DP_DELTA):
        eps_ablation_results[level] = _main
        print(f"[{level:8s}] eps={target:g}: reusing the §24.7 run from memory")
        _save_level(level, _main)
        continue

    # 2) a previously saved ablation level
    _res = _load_run(f"noniid_eps_ablation_{level}_history.csv",
                     f"noniid_eps_ablation_{level}_privacy.json", target)
    # 3) for the level matching §24.7, the files §25.2 saved
    if _res is None:
        _res = _load_run(MAIN_RUN_HISTORY, MAIN_RUN_PRIVACY, target)
    if _res is not None:
        eps_ablation_results[level] = _res
        print(f"[{level:8s}] eps={target:g}: loaded from {_res['source']}")
        continue

    # 4) train it, if this session is allowed to
    if level not in EPS_LEVELS_TO_RUN:
        print(f"[{level:8s}] eps={target:g}: not on disk and not in EPS_LEVELS_TO_RUN; skipped")
        continue
    print(f"\n=== [{level}] DP-FedAvg, target eps={target:g}, calibration={DP_CALIBRATION} ===")
    _res = run_dp_federated_experiment(
        clients_noniid, GLOBAL_TEST_SAMPLES, target_epsilon=target,
        tag=f"dp_{level}_eps{target:g}", seed=SEED)
    assert np.isfinite(_res["privacy"]["epsilon"]), f"non-finite epsilon for {level}"
    eps_ablation_results[level] = _res
    _save_level(level, _res)          # persist immediately
    print(f"[{level:8s}] saved to {OUT_DIR}")

print("\nLevels available:", [l for l in EPS_ABLATION if l in eps_ablation_results])


In [ ]:
# ---- epsilon ablation, part 2: table and figure from whatever levels exist --
_levels = [l for l in EPS_ABLATION if l in eps_ablation_results]
_missing = [l for l in EPS_ABLATION if l not in eps_ablation_results]
assert _levels, "no ablation level available yet -- run part 1 first"
if _missing:
    print(f"Not yet available (run them in a later session): {_missing}\n")

_base = fed_results["fedavg"]["final_metrics"]
_rows = [{
    "level": "non-private", "target_eps": np.inf, "epsilon": np.inf, "delta": np.nan,
    "sigma_min": 0.0, "sigma_max": 0.0, "clip_C": np.nan,
    "accuracy": _base["acc"], "balanced_acc": _base["balanced_acc"],
    "macro_f1": _base["f1_mac"], "auc": _base["auc"],
    "ece": _base["ece"], "brier": _base["brier"],
}]
for level in _levels:
    _p = eps_ablation_results[level]["privacy"]
    _m = eps_ablation_results[level]["final_metrics"]
    _rows.append({
        "level": level, "target_eps": EPS_ABLATION[level], "epsilon": _p["epsilon"],
        "delta": _p["delta"],
        "sigma_min": float(np.min(_p["noise_multiplier"])),
        "sigma_max": float(np.max(_p["noise_multiplier"])),
        "clip_C": _p["max_grad_norm"],
        "accuracy": _m["acc"], "balanced_acc": _m["balanced_acc"],
        "macro_f1": _m["f1_mac"], "auc": _m["auc"],
        "ece": _m["ece"], "brier": _m["brier"],
    })
_main = globals().get("dp_result")
if _main is not None and all(abs(_main["privacy"]["target_epsilon"] - t) > 1e-9 for t in EPS_ABLATION.values()):
    _mp, _mm = _main["privacy"], _main["final_metrics"]
    _rows.append({
        "level": "main (reference)", "target_eps": _mp["target_epsilon"], "epsilon": _mp["epsilon"],
        "delta": _mp["delta"], "sigma_min": float(np.min(_mp["noise_multiplier"])),
        "sigma_max": float(np.max(_mp["noise_multiplier"])), "clip_C": _mp["max_grad_norm"],
        "accuracy": _mm["acc"], "balanced_acc": _mm["balanced_acc"], "macro_f1": _mm["f1_mac"],
        "auc": _mm["auc"], "ece": _mm["ece"], "brier": _mm["brier"],
    })
eps_ablation_df = pd.DataFrame(_rows)
eps_ablation_df = pd.concat([eps_ablation_df.iloc[:1],
                             eps_ablation_df.iloc[1:].sort_values("target_eps")]).reset_index(drop=True)
for _col, _key in [("accuracy", "acc"), ("macro_f1", "f1_mac"), ("auc", "auc")]:
    eps_ablation_df[f"d_{_col}"] = eps_ablation_df[_col] - _base[_key]
eps_ablation_df.to_csv(OUT_DIR / "noniid_eps_ablation_summary.csv", index=False)

pd.set_option("display.width", 240)
print("Privacy-level ablation (epsilon from the accountant after training; "
      "d_* = change vs non-private FedAvg, negative = worse):")
print(eps_ablation_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

_eps = [eps_ablation_results[l]["privacy"]["epsilon"] for l in _levels]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

ax = axes[0]
for _col, _key, _mk in [("accuracy", "acc", "o"), ("macro_f1", "f1_mac", "s"), ("auc", "auc", "^")]:
    _vals = [eps_ablation_results[l]["final_metrics"][_key] for l in _levels]
    _line, = ax.plot(_eps, _vals, marker=_mk, linewidth=2, label=_col)
    ax.axhline(_base[_key], ls="--", linewidth=1, color=_line.get_color(), alpha=0.6)
ax.set_xscale("log")
ax.set_xticks(_eps)
ax.set_xticklabels([f"{l}\n{e:.2f}" for l, e in zip(_levels, _eps)], fontsize=8)
ax.minorticks_off()
ax.set_xlabel("privacy level / epsilon (log scale)"); ax.set_ylabel("score")
ax.set_title("Utility vs privacy budget (dashed = non-private FedAvg)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
_bh = histories.get("fedavg", {}) if "histories" in globals() else {}
if _bh.get("round"):
    ax.plot(_bh["round"], _bh["acc"], "k--", linewidth=1.6, label="non-private FedAvg")
else:
    ax.axhline(_base["acc"], ls="--", color="k", linewidth=1.2,
               label="non-private FedAvg (recorded final)")
for l, e in zip(_levels, _eps):
    _h = eps_ablation_results[l]["history"]
    ax.plot(_h["round"], _h["acc"], linewidth=2, label=f"{l} (eps={e:.2f})")
ax.set_xlabel("Communication round"); ax.set_ylabel("Global test accuracy")
ax.set_title("Convergence by privacy level"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig_noniid_eps_ablation.png", dpi=150, bbox_inches="tight")
plt.show()


# §26 · DP Section — Scope and Limitations

**What the epsilon covers.** Sample-level DP for each client's own training
records, accumulated over all local steps that client takes across the 50
rounds, under Poisson subsampling with the RDP accountant. It is *not*
client-level DP: a hospital's participation is not hidden, only the influence
of any single scan within its own dataset. The headline figure is the maximum
over clients, so it holds for every record in the federation.

**Non-IID makes the privacy cost uneven.** Under Dirichlet(alpha=0.3) the
clients differ by more than an order of magnitude in size. A small client has a
high sampling rate `q = 1/ceil(n/batch)`, so privacy amplification by subsampling
barely applies to it and it needs substantially more noise for the same
epsilon — while simultaneously contributing the fewest local steps and the
smallest aggregation weight. The per-client table in §25 makes this explicit.
This is a property of DP under quantity skew, not a bug, and it is the main
reason a non-IID DP arm degrades more than an IID one.

**Batch size is unchanged.** `BATCH_SIZE = 32` is kept from the baseline even
though it is a large fraction of the smallest client's data. Shrinking the
batch for small clients would lower their `q` and reduce the noise they need,
but it would also change the baseline's training configuration; the run warns
where `q > 0.2` instead.

**Class weights are computed over the pooled label histogram.** This matches
the baseline exactly, and is kept for comparability — but under the DP threat
model it is an un-accounted-for release: the global class frequencies are a
statistic of all clients' private labels and they influence the released model
without passing through the noise mechanism. Set
`DP_UNIFORM_CLASS_WEIGHTS = True` to remove the leak at the cost of no longer
matching the baseline's loss.

**Deviations from the non-private arm** (all forced, all reported above):
BatchNorm -> GroupNorm; `multiscale.scale_weight` frozen (3 params); AMP off;
weighted-CE renormalized to an exact per-sample mean; prototype-margin term
disabled. FedProx has no valid DP-SGD counterpart, so the private arm is
compared against FedAvg; the server-side aggregation itself
(`federated_aggregate`, sample-count-weighted averaging) is unchanged.

**Not cryptographically secure.** `secure_mode=False`, so the Gaussian noise
comes from PyTorch's standard RNG rather than a cryptographically secure
source — this is what keeps the run reproducible from `SEED`. Standard for
research reporting; state it if the paper claims a deployment guarantee.

**Aggregation is in the clear.** The server sees each client's noised update.
DP-SGD bounds what those updates reveal about individual records; it is not a
substitute for secure aggregation.

**This is Stage-2 formal privacy.** It is unrelated to and must not be
described alongside the Stage-1 Gaussian input-perturbation robustness /
uncertainty experiment.

**Class weights are outside the accountant.** `omega_c` is computed once from
the pooled label counts of all clients, is not noised, and is not part of the
privacy accounting. The reported epsilon therefore covers the per-sample
gradient computations only.

**Calibration of the noise.** Since this revision the noise multiplier is solved
with the sampling rate and the number of steps per epoch read from the Opacus
`DPDataLoader` that actually trains each client, and the run asserts the match in
round 1 (`solver_matches_loader`). Earlier runs used `BATCH_SIZE / n_train` and
`floor(n_train / BATCH_SIZE)`; for the 57-image client that meant 1 step per epoch at
q = 0.561 against 2 steps at q = 0.5, and an achieved epsilon of 10.51 against a target
of 8. Those runs are not comparable and are not reused by the ablation.

**What the controls separate.** `RUN_GN_CONTROL` isolates normalization, loss and
AMP changes from the privacy mechanism; `RUN_CLIP_ONLY_CONTROL` isolates
per-sample clipping from the noise. Neither run is private.
